# RAG regulasi pajak

## 1. Setup

### 1.1 Impor & pembaca .env

In [2]:
import json
import os
import re
import time
from pathlib import Path

import requests

BAWAAN = {
    "": ("http://192.168.18.185:1234/v1", "google/gemma-4-12b-qat"),
    "groq": ("https://api.groq.com/openai/v1", "llama-3.3-70b-versatile"),
}


def muat_env(f=Path.cwd() / ".env"):
    if not f.exists():
        print(f"!! {f} tidak ada -- pakai bawaan")
        return {}
    env = {}
    for baris in f.read_text(encoding="utf-8").splitlines():
        baris = baris.strip()
        if baris and not baris.startswith("#") and "=" in baris:
            k, _, v = baris.partition("=")
            env[k.strip()] = v.strip()
    return env


ENV = muat_env()


def sumber(nama=""):
    e = {k.upper(): v for k, v in ENV.items()}
    s = f"_{nama}".upper() if nama else ""
    base_bawaan, model_bawaan = BAWAAN.get(nama, ("", ""))
    return (e.get(f"LLM_BASE{s}") or e.get(f"LLM{s}") or base_bawaan,
            e.get(f"LLM_MODEL{s}") or model_bawaan,
            e.get(f"LLM_KEY{s}", ""))


def _siap(nama=""):
    """-> (endpoint chat, nama model, kunci). Tidak mengubah apa pun."""
    base, model, kunci = sumber(nama)
    assert base and model, (
        f"penyedia {nama or 'lokal'!r} tidak dikenal. Isi LLM_BASE{'_' + nama if nama else ''}"
        f" dan LLM_MODEL{'_' + nama if nama else ''} di .env, atau tambahkan ke BAWAAN.")
    return base.rstrip("/") + "/chat/completions", model, kunci


def pakai(nama=""):
    """Ganti penyedia DEFAULT -- yang dipakai panggilan yang tidak menyebut
    penyedianya sendiri. -> nama modelnya."""
    global LLM, MODEL_LLM, LLM_KEY
    LLM, MODEL_LLM, LLM_KEY = _siap(nama)
    print(f"default -> {MODEL_LLM}")
    return MODEL_LLM


### 1.2 Knop penyedia
Satu tempat untuk semua sel di bawah -- tidak ada sel yang diam-diam pakai model lain.

In [3]:
ENV = muat_env()

# Satu-satunya tempat memilih model. Sel coba-coba di bawah ikut ini, kecuali
# yang menimpanya sendiri lewat PAKAI_PLANNER / PAKAI_JAWAB.
MODEL_PLANNER = ""            # "" = lokal
MODEL_PENJAWAB = "Groq"

print("penyedia di .env:", ", ".join(
    repr(n) for n in sorted({""} | {k.upper()[len("LLM_MODEL_"):].lower()
                                    for k in ENV if k.upper().startswith("LLM_MODEL_")})))
for _tahap, _n in (("planner ", MODEL_PLANNER), ("penjawab", MODEL_PENJAWAB)):
    base, model, kunci = sumber(_n)
    tanda = "" if base and model else "   << TIDAK LENGKAP"
    print(f"{_tahap} : {model or '?':32} @ {base or '?'}"
          f"  kunci {'ada' if kunci else 'kosong'}{tanda}")

pakai(MODEL_PLANNER)

penyedia di .env: '', 'groq', 'openai'
planner  : google/gemma-4-12b-qat           @ http://192.168.18.185:1234/v1  kunci kosong
penjawab : openai/gpt-oss-120b              @ https://api.groq.com/openai/v1  kunci ada
default -> google/gemma-4-12b-qat


'google/gemma-4-12b-qat'

## 2. System prompt Planner

In [4]:
SISTEM_PLANNER = """# ROLE

Kamu adalah Planner Agent dalam sistem RAG regulasi pajak Indonesia.
Tugasmu HANYA menganalisis pertanyaan user dan menghasilkan rencana
retrieval terstruktur. Kamu TIDAK menjawab pertanyaan secara langsung,
dan TIDAK menjelaskan isi peraturan apapun.

Output kamu HARUS berupa JSON valid saja, tanpa teks lain, tanpa
markdown code fence, tanpa preamble.


# LANGKAH KERJA

Tugasmu MEMBACA pertanyaan. Kamu TIDAK memilih metode pencarian, TIDAK
menentukan filter, dan TIDAK memutuskan perlu-tidaknya penelusuran versi --
semua itu diturunkan sistem dari jawabanmu. Isi field sesuai urutannya:

1. Catat SEMUA peraturan & pasal yang DISEBUT pengguna (apa adanya)
2. Tentukan maksud waktunya
3. Baru klasifikasikan tipe pertanyaannya
4. Tulis ulang DAN perjelas pertanyaan ke bahasa peraturan (rewritten_query)
5. Pecah jadi sub_queries -- WAJIB, 2 sampai 4 butir
6. Perkirakan cakupannya -- berapa BANYAK tempat yang harus dibaca

Urutan 1 sebelum 3 itu disengaja: lihat dulu apa yang tertulis, baru
menyimpulkan jenisnya.


# TIPE PERTANYAAN

Pilih SATU. Ini menentukan cara sistem mencari, jadi pilih berdasar APA yang
ditanya -- bukan bagaimana pengguna menuliskannya. Pertanyaan berbahasa
sehari-hari tetap punya tipe yang sama dengan versi formalnya.

- definisi-otoritatif: menanyakan arti sebuah istilah, DAN peraturannya
  disebut. Jawabannya tunggal.
- definisi-tabrakan: menanyakan arti sebuah istilah TANPA mengaitkannya ke
  satu peraturan -- istilah yang sama sering didefinisikan berbeda di banyak
  dokumen, dan pengguna perlu melihat perbedaannya.
- tarif: jawabannya angka atau persentase.
- prosedur: tata cara, syarat, batas waktu, siapa yang wajib.
- multi-dokumen: butuh >=2 unit dari dokumen BERBEDA untuk jawaban utuh
  (termasuk relasi UU-PP-PMK turunan).
- bernomor: pengguna menyebut nomor peraturan atau nomor pasal, dan yang
  diminta isi/identitas peraturan itu sendiri.
- perubahan: jawabannya ada di peraturan PENGUBAH, bukan di peraturan asal.

# YANG DICATAT DARI PERTANYAAN

Cari dan salin APA ADANYA -- jangan menebak, jangan melengkapi, jangan
menormalkan:

- peraturan         : SEMUA peraturan yang disebut pengguna, urut kemunculan.
                      ["PMK 231/2019", "PMK 59/2022"] -- jangan berhenti di yang
                      pertama, dan jangan memilih salah satu.
- pasal             : SEMUA alamat pasal yang disebut, nomornya saja --
                      ["57"], ["4", "6"]. Satu pun tetap ditulis sebagai daftar.
                      DIKTUM ikut di sini. Dokumen berbentuk penetapan
                      (Keputusan) tidak memakai Pasal melainkan diktum
                      berurut: "Diktum keempat belas" -> ["keempat belas"].
                      Salin apa adanya, JANGAN diubah jadi angka.
- tahun             : tahun/tanggal spesifik yang disebut pengguna

Istilah pajak yang muncul ("NITKU", "Faktur Pajak", "PPh 22") tidak punya
field sendiri, tapi tetap harus kamu perhatikan: singkatannya dipanjangkan di
rewritten_query dan jadi bahan sub_queries.


## Mana yang MASUK daftar, mana yang tidak

Ujinya satu, dan diterapkan ke tiap frasa satu per satu: apakah frasa itu
menunjuk SATU peraturan tertentu?

  "Pasal 9 PMK 231/2019 setelah diubah PMK 59/2022"
                                  -> ["PMK 231/2019", "PMK 59/2022"]
        DUA-DUANYA, dan urut seperti disebut. Untuk pertanyaan semacam ini
        jawabannya justru ada di peraturan yang KEDUA -- mencatat yang
        pertama saja membuang satu-satunya dokumen yang memuatnya.

  "UU KUP"                        -> ["UU KUP"]
        nama resmi satu undang-undang. Tanpa nomor pun tetap dicatat --
        sistem yang mencarikan nomornya, JANGAN kamu tebak.

  "PP 55 tahun 2022"              -> ["PP 55 tahun 2022"]

  "diatur di PMK tersendiri"      -> []
        "PMK" itu JENIS peraturan, bukan nama satu peraturan -- dan
        penggunanya justru sedang bilang dia tidak tahu yang mana.

  "menurut ketentuan perpajakan"  -> []

Pasal juga dicatat SEMUANYA, dengan alasan yang sama seperti peraturan:

  "Pasal 57 PP 55/2022 dan Pasal 9 PMK 231/2019"  -> pasal ["57", "9"]
  "Pasal 4 dan Pasal 6 UU PPh"                    -> pasal ["4", "6"]

Berhenti di pasal pertama membuat pasal kedua tidak pernah dicari, dan itu
tidak menimbulkan error apa pun -- pencarian tetap berhasil, jawabannya saja
yang separuh.

Kalau ragu, JANGAN dimasukkan. Satu butir yang salah membuat pencarian
disaring ke peraturan yang keliru: hasilnya nol, tanpa error. Daftar yang
lebih pendek cuma membuat pencarian lebih luas.

Peraturan PEMBANDING tetap dicatat. Menyaring atau tidak itu keputusan
sistem, bukan keputusanmu: untuk pertanyaan yang memang ingin melihat
beberapa dokumen sekaligus, sistem sendiri yang mengosongkan saringannya.


# REWRITTEN_QUERY (selalu diisi)

DUA tugas sekaligus, bukan satu:

1. TERJEMAHKAN. Istilah sehari-hari jadi istilah resmi, singkatan
   dipanjangkan. "telat lapor" -> "keterlambatan penyampaian Surat
   Pemberitahuan".
2. PERJELAS. Ganti sebutan sehari-hari jadi istilah resmi, lalu tulis
   aspek yang ditanya sebagai kalimat utuh.

Yang ditulis pengguna BOLEH dipakai semua -- termasuk angka yang dia sebut.
Yang dilarang cuma tiga hal di bawah ini.


## A. JANGAN menambah yang tidak disebut

Kalau misal pengguna bilang "omzet 400 juta setahun", kamu TIDAK tahu dia orang
pribadi atau badan, sudah PKP atau belum, tahun pajak berapa. Menuliskan
tebakan itu membuat seluruh pencarian tertarik ke arah yang mungkin salah
sejak langkah pertama, dan tidak ada tahap sesudahnya yang bisa
membetulkannya. intinya jangan menambahkan asumsi yang tidak disebut pengguna.

Kalau ada beberapa bacaan yang bersaing, jangan pilih salah satu di
rewritten_query. Taruh sebagai butir terpisah di sub_queries, biar bukti dari
korpus yang memutuskan.


## B. JANGAN menggeser aspek yang ditanya

Ini kesalahan paling halus, dan paling sering. Yang ditanya harus tetap yang
ditanya -- definisi tetap definisi, tarif tetap tarif, daftar tetap daftar.

  "materai itu dipakai buat dokumen apa aja"

    BENAR -> "dokumen yang dikenai Bea Meterai"
    SALAH -> "tarif Bea Meterai"

Perhatikan: dua-duanya memakai bahan yang sama persis, tidak ada yang
ditambah. Bedanya cuma ASPEK -- dan itu sudah cukup untuk merusak. Yang
ditanya DOKUMEN APA SAJA, tapi versi SALAH menyeret
pencarian ke BERAPA TARIFNYA (Pasal 5). Jawabannya jadi nyerempet benar --
dan itu justru yang bikin salahnya sulit terlihat.

Kalau pengguna menyebut angkanya sendiri ("materai 10 ribu"), angka itu boleh
ikut. Yang dilarang tetap sama: menggeser aspeknya.


## C. JANGAN mengoreksi angka pengguna

Walau kamu tahu angka yang berlaku sekarang berbeda.

  "materai 6000 dipakai untuk apa?"   -> tulis tetap 6000, jangan Rp10.000

Tarif Bea Meterai sebelum UU 10/2020 memang Rp3.000 dan Rp6.000. Angka itu
sering satu-satunya petunjuk bahwa pengguna bertanya soal aturan LAMA.
"Membetulkannya" menghapus petunjuk itu, dan jawabannya jadi dari zaman yang
salah tanpa ada yang menyadari.


# SUB_QUERIES -- WAJIB, 2 SAMPAI 4 BUTIR (sebutuhnya saja. lebih dari 4, gabungkan dulu ke 4 butir yang paling relevan)

Pecah rewritten_query jadi beberapa kunci pencarian. TIDAK PERNAH kosong,
TIDAK PERNAH cuma 1. Pertanyaan sesederhana apapun tetap dipecah.

Alasannya: korpus dipotong kecil-kecil -- satu pasal, satu ayat, atau satu
angka definisi. Satu kunci pencarian cuma menjangkau satu titik. Beberapa
kunci menjangkau beberapa titik, hasilnya digabung.

Ada DUA cara memecah. Pilih sesuai pertanyaannya.

## A. PECAH PER ASPEK

Dipakai kalau pertanyaan menuntut lebih dari satu hal yang letaknya
kemungkinan besar di pasal berbeda. Penanda: kata sambung "dan", "lalu",
"serta" dan sejenisnya, atau lebih dari satu tanda tanya.

Aspek yang biasanya terpisah pasal:
definisi | fungsi atau kegunaan | tarif/besaran | syarat | tata cara |
batas waktu | sanksi | pengecualian | siapa yang wajib

  "NPPN itu apa dan dipakai untuk menghitung apa?"
  -> ["definisi Norma Penghitungan Penghasilan Neto",
      "penggunaan Norma Penghitungan Penghasilan "]

## B. PECAH PER SISI

Dipakai kalau pertanyaan cuma menyangkut SATU aspek.

JANGAN menulis ulang hal yang sama dengan kata berbeda. Pertanyaan asli
pengguna sudah ikut dicari otomatis di luar rencanamu, jadi "pengertian
Surat Paksa" dan "Surat Paksa adalah" tidak menjangkau titik baru --
tiga kunci, satu titik, dua jatah terbuang.

Yang benar: tulis sisi-sisi lain yang harus diketahui supaya jawabannya
utuh. Butir PERTAMA tetap inti pertanyaannya.

Sisi yang biasanya ada di teks peraturan:
siapa yang menerbitkan | kapan diterbitkan | apa akibat hukumnya |
apa isinya | dasar hukumnya

  "Apa itu Surat Paksa?"
  -> ["definisi Surat Paksa",
      "pihak yang menerbitkan Surat Paksa",
      "kekuatan hukum Surat Paksa"]

Dua batas, dua-duanya keras:

1. SISI lain boleh, TOPIK lain tidak. "pihak yang menerbitkan Surat Paksa"
   masih tentang Surat Paksa. "sanksi keterlambatan pajak" topik lain yang
   kebetulan bertetangga -- itu menyeret pasal yang tidak diminta.

2. Sebut SLOT-nya, jangan diisi.

     BENAR -> "pihak yang menerbitkan Surat Paksa"
     SALAH -> "Surat Paksa diterbitkan oleh Pejabat"

   Versi SALAH sudah menjawab duluan. Kalau tebakanmu meleset, pencarian
   ditarik ke arah yang salah sejak awal dan tidak ada tahap sesudahnya yang
   bisa membetulkannya. Kamu boleh meniru BENTUK kalimat peraturan; kamu
   tidak boleh mengisi NILAI yang belum kamu lihat.

## Cara menulis tiap butir

- Berbunyi seperti KALIMAT PERATURAN, bukan seperti pertanyaan orang.
  Jangan pakai "apa", "bagaimana", "berapa", atau tanda tanya.
- Singkatan DIPANJANGKAN ("NPPN" -> "Norma Penghitungan Penghasilan
  Neto"), karena teks peraturan memakai bentuk panjangnya.
- Kalau pengguna menyebut pasal tertentu, nomor pasalnya ikut ditulis.
- JANGAN menyalin ulang pertanyaan asli apa adanya. Pertanyaan asli sudah
  ikut dicari otomatis di luar rencanamu -- kalau kamu salin lagi, satu
  jatah pencarian terbuang percuma.


# MAKSUD WAKTU -- CARA MENENTUKAN

Bacalah maksudnya, jangan kata kuncinya. "mulai terutang sejak kapan?"
menanyakan KAPAN ATURANNYA BERLAKU -- itu isi pasal, bukan permintaan versi
terbaru, jadi modenya tetap "latest".

Tahun yang jadi BAGIAN NAMA peraturan bukan maksud waktu. "PP 55 tahun 2022"
itu nama dokumennya -- penanyanya tidak sedang minta bunyi versi 2022. Modenya
"latest", dan tahun dibiarkan kosong.

- "latest" (default): tidak ada penanda waktu, atau penandanya justru menunjuk
  keadaan sekarang -- "sekarang", "saat ini", "yang berlaku".
- "as_of": penanya menyebut TITIK WAKTU yang bukan bagian nama peraturan --
  "bunyi Pasal 1 pada 2013", "aturan yang berlaku waktu itu", "materai 6000".
  -> isi tahun dengan tahun yang dimaksud. Sistem mengirim SATU versi: yang
  berlaku pada tahun itu.
- "riwayat": yang ditanya STATUS perubahannya, bukan bunyinya -- "sudah pernah
  diubah belum", "berubah berapa kali", "kapan terakhir direvisi", "masih
  berlaku?". Sistem cukup mengirim daftar perubahannya, tanpa teks pasal.
- "diff": penanya membandingkan DUA keadaan, sebelum lawan sesudah satu
  perubahan tertentu -- "setelah diubah PMK 59/2022 jadi apa", "bedanya
  sebelum dan sesudah".
- "all": penanya minta melihat SELURUH perjalanan pasalnya. Jarang, dan hanya
  kalau memang diminta -- "semua versi", "dari awal sampai sekarang".

Kalau ragu antara "as_of" dan "latest", pilih "latest". Kalau ragu antara
"riwayat" dan "diff", pilih "diff" -- dia tetap membawa bunyi pasalnya.


# CAKUPAN -- SEBERAPA BANYAK BAHAN YANG DIBUTUHKAN

Yang diukur: jawabannya tersebar di BERAPA TEMPAT -- bukan seberapa panjang
jawabannya, bukan seberapa sulit pertanyaannya. Ini menentukan berapa potongan
peraturan yang diambil sistem. Kekecilan bikin jawaban bolong; kebesaran
menenggelamkan yang penting di antara potongan yang tidak terpakai.

- "sempit": jawabannya ada di SATU pasal yang sudah bisa ditunjuk. Biasanya
  pengguna menyebut sendiri nomor pasalnya.
    "Apa isi Pasal 57 PP 55 tahun 2022?"
    "Bunyi Pasal 9 PMK 231/2019 setelah diubah PMK 59/2022"

- "sedang" (default): satu topik, mungkin tersebar di beberapa pasal
  berdekatan -- definisi beserta syaratnya, tarif beserta pengecualiannya.
    "Apa itu Surat Paksa?"
    "syarat menjadi Pengusaha Kena Pajak"

- "luas": jawabannya berupa DAFTAR, atau harus dirakit dari beberapa dokumen
  yang berbeda.
    "Siapa saja yang wajib memungut PPh Pasal 22?"
    "Batas akhir lapor SPT Tahunan orang pribadi dan badan"

Kalau ragu antara dua, ambil yang lebih LUAS. Jawaban bolong lebih merugikan
daripada bahan yang sedikit berlebih.


# ATURAN PENGISIAN FIELD KOSONG

Field bertipe string yang tidak berlaku diisi STRING KOSONG "", bukan null.
Array yang tidak berlaku diisi array kosong [].
Kecuali sub_queries: itu TIDAK BOLEH kosong, minimal 2 butir.


# OUTPUT SCHEMA (WAJIB, JSON SAJA)

{
  "peraturan": ["array of string, apa adanya dari pengguna, atau []"],
  "pasal": ["array of string, nomor pasal saja (\"57\", \"31E\"), atau []"],
  "temporal_mode": "latest | as_of | riwayat | diff | all",
  "tahun": "string, diisi hanya kalau temporal_mode = as_of",
  "tipe": "string, salah satu dari 7 tipe di atas",
  "cakupan": "sempit | sedang | luas",
  "rewritten_query": "string, WAJIB diisi",
  "sub_queries": ["array of string, WAJIB 2-4 butir"]
}

# CONTOH

Q: "Apa isi Pasal 57 PP 55 tahun 2022?"
{
  "peraturan": ["PP 55 tahun 2022"],
  "pasal": ["57"],
  "temporal_mode": "latest",
  "tahun": "",
  "tipe": "bernomor",
  "cakupan": "sempit",
  "rewritten_query": "ketentuan yang diatur dalam Pasal 57 Peraturan Pemerintah Nomor 55 Tahun 2022",
  "sub_queries": [
    "Pasal 57 Peraturan Pemerintah Nomor 55 Tahun 2022",
    "ketentuan yang diatur dalam Pasal 57"
  ]
}

Q: "materai 10 ribu itu dipakai buat dokumen apa aja?"
{
  "peraturan": [],
  "pasal": [],
  "temporal_mode": "latest",
  "tahun": "",
  "tipe": "prosedur",
  "cakupan": "luas",
  "rewritten_query": "dokumen yang dikenai Bea Meterai",
  "sub_queries": [
    "dokumen yang dikenai Bea Meterai",
    "dokumen yang dikecualikan dari Bea Meterai",
    "saat terutang Bea Meterai"
  ]
}

Q: "Bunyi Pasal 9 PMK 231/2019 setelah diubah PMK 59/2022 seperti apa?"
{
  "peraturan": ["PMK 231/2019", "PMK 59/2022"],
  "pasal": ["9"],
  "temporal_mode": "diff",
  "tahun": "",
  "tipe": "perubahan",
  "cakupan": "sempit",
  "rewritten_query": "bunyi Pasal 9 Peraturan Menteri Keuangan Nomor 231/PMK.03/2019 setelah diubah dengan Peraturan Menteri Keuangan Nomor 59/PMK.03/2022",
  "sub_queries": [
    "Pasal 9 Peraturan Menteri Keuangan Nomor 231/PMK.03/2019",
    "ketentuan Pasal 9 diubah sehingga berbunyi",
    "perubahan Pasal 9 dalam Peraturan Menteri Keuangan Nomor 59/PMK.03/2022"
  ]
}

Q: "Pasal 1 PER-57/PJ/2010 sudah pernah diubah belum?"
{
  "peraturan": ["PER-57/PJ/2010"],
  "pasal": ["1"],
  "temporal_mode": "riwayat",
  "tahun": "",
  "tipe": "perubahan",
  "cakupan": "sempit",
  "rewritten_query": "riwayat perubahan Pasal 1 Peraturan Direktur Jenderal Pajak Nomor PER-57/PJ/2010",
  "sub_queries": [
    "Pasal 1 Peraturan Direktur Jenderal Pajak Nomor PER-57/PJ/2010",
    "ketentuan Pasal 1 diubah sehingga berbunyi"
  ]
}
"""

## 3. Skema keluaran

In [5]:

TIPE = ["definisi-otoritatif", "definisi-tabrakan", "tarif", "prosedur",
        "multi-dokumen", "bernomor", "perubahan"]

SUB_MIN, SUB_MAX = 2, 4

# Berapa potongan peraturan yang diambil. Bukan "seberapa sulit pertanyaannya",
# tapi "jawabannya tersebar di berapa tempat": satu pasal yang ditunjuk sendiri
# oleh penanya butuh 3, daftar pemungut PPh 22 yang berserak butuh 9.
CAKUPAN = {"sempit": 3, "sedang": 6, "luas": 9}

# Maksud waktu -> berapa versi yang benar-benar dikirim. Lima jalur, bukan dua:
# "bunyi pada 2013" butuh SATU versi (yang berlaku waktu itu), dan "sudah pernah
# diubah belum" tidak butuh teks pasal sama sekali -- cukup daftar perubahannya.
MODE = ["latest", "as_of", "riwayat", "diff", "all"]

def obj(props):
    """Objek strict: semua properti wajib, tidak boleh ada properti liar."""
    return {"type": "object", "properties": props,
            "required": list(props), "additionalProperties": False}

STR, BOOL = {"type": "string"}, {"type": "boolean"}
ARR_STR = {"type": "array", "items": STR}

SKEMA_PLANNER = {
    "type": "json_schema",
    "json_schema": {
        "name": "pemahaman_pertanyaan",
        "strict": True,
        "schema": obj({
            "peraturan": {**ARR_STR, "maxItems": 5},
            "pasal": {**ARR_STR, "maxItems": 4},
            "temporal_mode": {"type": "string", "enum": MODE},
            "tahun": STR,
            "tipe": {"type": "string", "enum": TIPE},
            "cakupan": {"type": "string", "enum": list(CAKUPAN)},
            "rewritten_query": STR,
            "sub_queries": {"type": "array", "items": STR,
                            "minItems": SUB_MIN, "maxItems": SUB_MAX},
        }),
    },
}

print("field:", list(SKEMA_PLANNER["json_schema"]["schema"]["properties"]))

field: ['peraturan', 'pasal', 'temporal_mode', 'tahun', 'tipe', 'cakupan', 'rewritten_query', 'sub_queries']


## 4. Pemanggil LLM

In [6]:
def _bersih(isi):
    return re.sub(r"<think>.*?</think>", "", isi, flags=re.S).strip()


def _json(isi):
    isi = _bersih(isi)
    try:
        return json.loads(isi)
    except json.JSONDecodeError:
        m = re.search(r"\{.*\}", isi, re.S)  # kadang masih terbungkus prosa
        if not m:
            raise
        return json.loads(m.group())


_TOLAK_SKEMA = set()

SETELAN = {
    "gpt-5":  {},                    # generasi baru: temperature harus default
    "gemma":  {"temperature": 0},
    "llama":  {"temperature": 0},
    "qwen":   {"temperature": 0},
    "":       {"temperature": 0},    # bawaan untuk model yang belum terdaftar
}

_TOLAK_TEMP = set()


def setelan(nama):
    """-> field tambahan untuk badan permintaan model ini."""
    n = nama.lower()
    for potongan, isi in SETELAN.items():
        if potongan and potongan in n:
            return dict(isi)
    return dict(SETELAN[""])


def panggil_llm(sistem, skema, pertanyaan, timeout=420, penyedia=None):
    """skema=None -> jawaban dikembalikan sebagai teks biasa, bukan JSON.
    Penjawab tidak punya bentuk tetap, jadi memaksanya jadi JSON cuma menambah
    satu cara gagal tanpa menambah apa pun.

    penyedia=None -> pakai yang sedang aktif (lihat pakai()).
    penyedia="groq" -> paksa penyedia itu UNTUK PANGGILAN INI SAJA. Dipakai
    supaya planner dan penjawab bisa jalan di model berbeda tanpa saling
    menimpa setelan global.
    """
    ujung, nama, kunci = (LLM, MODEL_LLM, LLM_KEY) if penyedia is None else _siap(penyedia)
    badan = {
        "model": nama,
        "messages": [{"role": "system", "content": sistem},
                    {"role": "user", "content": pertanyaan}],
    }
    if nama not in _TOLAK_TEMP:
        badan.update(setelan(nama))
    if skema:
        badan["response_format"] = {"type": "json_object"} if nama in _TOLAK_SKEMA else skema
    kepala = {"Authorization": f"Bearer {kunci}"} if kunci else {}
    t0 = time.perf_counter()
    for percobaan in range(4):
        try:
            r = requests.post(ujung, timeout=timeout, json=badan, headers=kepala)
        except requests.exceptions.RequestException as e:
            if percobaan == 3:
                raise RuntimeError(
                    f"{nama} tidak menjawab setelah 4 percobaan "
                    f"(timeout {timeout}s): {type(e).__name__}") from e
            print(f"  {type(e).__name__} -> ulangi ({percobaan + 2}/4)")
            continue
        if r.status_code != 429:
            break
        jeda = float(r.headers.get("retry-after", 2 * (percobaan + 1)))
        print(f"  429, tunggu {jeda}s")
        time.sleep(jeda)
    if r.status_code == 400 and "temperature" in r.text:
        print(f"  {nama} tolak temperature -> tidak dikirim seterusnya")
        _TOLAK_TEMP.add(nama)
        badan.pop("temperature", None)
        r = requests.post(ujung, timeout=timeout, json=badan, headers=kepala)
    if r.status_code == 400 and "json_schema" in r.text:
        print(f"  {nama} tolak json_schema -> pakai json_object seterusnya")
        _TOLAK_SKEMA.add(nama)
        badan["response_format"] = {"type": "json_object"}
        r = requests.post(ujung, timeout=timeout, json=badan, headers=kepala)
    if not r.ok:  # pesan server ikut dibawa; "400 Bad Request" saja tidak menjelaskan apa pun
        raise RuntimeError(f"{nama} tolak permintaan ({r.status_code}): {r.text[:400]}")
    isi = r.json()["choices"][0]["message"]["content"]
    return (_json(isi) if skema else _bersih(isi)), time.perf_counter() - t0, nama

#### cek: setelan per model

In [7]:
def _uji_setelan():
    """Salah petak di sini tidak bikin error -- cuma diam-diam mengirim
    temperature ke model yang menolaknya, dan itu 400 di tengah eval."""
    assert setelan("gpt-5.6-luna") == {}, setelan("gpt-5.6-luna")
    assert setelan("google/gemma-4-12b-qat") == {"temperature": 0}
    assert setelan("llama-3.3-70b-versatile") == {"temperature": 0}
    assert setelan("model-yang-belum-ada") == {"temperature": 0}
    # dict yang dikembalikan harus salinan; kalau tidak, satu panggilan yang
    # menambah field ke badan ikut mengubah SETELAN untuk panggilan berikutnya
    x = setelan("gemma"); x["temperature"] = 9
    assert SETELAN["gemma"] == {"temperature": 0}, SETELAN["gemma"]
    print("ok  setelan per model (4 kasus + salinan)")


_uji_setelan()

ok  setelan per model (4 kasus + salinan)


### 4.1 planner() -- panggil LLM, lengkapi & periksa keluarannya

In [8]:
BENTUK = {
    "peraturan": [], "pasal": [], "temporal_mode": "latest", "tahun": "",
    "tipe": "", "cakupan": "sedang", "rewritten_query": "", "sub_queries": [],
}


def _lengkapi(hasil, bentuk=BENTUK, jalur=""):
    """Isi field yang tidak dikirim model dengan default, dan laporkan yang mana."""
    for k, v in bentuk.items():
        if k not in hasil:
            print(f"  !! field hilang: {jalur}{k} -> default {v!r}")
            hasil[k] = json.loads(json.dumps(v))  # salinan, jangan bagi objek default
        elif isinstance(v, dict):
            _lengkapi(hasil[k], v, f"{jalur}{k}.")
    return hasil


def planner(pertanyaan, penyedia=None):
    """-> pemahaman model + strategi yang DITURUNKAN darinya."""
    paham, detik, nama = panggil_llm(SISTEM_PLANNER, SKEMA_PLANNER, pertanyaan, penyedia=penyedia)
    paham = _lengkapi(paham)

    if not paham["rewritten_query"]:
        print("  !! rewritten_query kosong")
    # aturan 2-4 cuma dipaksa server kalau json_schema jalan; kalau tidak, di sini
    if not (SUB_MIN <= len(paham["sub_queries"]) <= SUB_MAX):
        print(f"  !! sub_queries {len(paham['sub_queries'])} butir, harusnya {SUB_MIN}-{SUB_MAX}")
    if not paham["sub_queries"]:  # jangan biarkan tahap retrieval jalan tanpa kunci
        paham["sub_queries"] = [paham["rewritten_query"] or pertanyaan]
        print(f"  !! sub_queries kosong -> dipakai {paham['sub_queries'][0]!r}")
    # json_object tidak memaksa tipe: kalau model mengirim string, iterasinya
    # jadi per-huruf dan cari_dokumen dipanggil untuk "P", "M", "K", ...
    for f in ("peraturan", "pasal"):
        if isinstance(paham[f], str):
            paham[f] = [paham[f]] if paham[f] else []
            print(f"  !! {f} dikirim sebagai string -> {paham[f]}")
    if paham["tipe"] not in TIPE:
        print(f"  !! tipe di luar daftar: {paham['tipe']!r}")
    if paham["temporal_mode"] not in MODE:   # -> pilih_versi() jatuh ke "latest"
        print(f"  !! temporal_mode di luar daftar: {paham['temporal_mode']!r}")
    if paham["cakupan"] not in CAKUPAN:   # -> turunkan() jatuh ke "sedang"
        print(f"  !! cakupan di luar daftar: {paham['cakupan']!r}")

    paham["strategi"] = turunkan(paham)   # <- keputusan, dihitung bukan diminta
    paham["_detik"] = round(detik, 2)
    paham["_model"] = nama
    return paham

def kunci_pencarian(paham, pertanyaan):
    return [pertanyaan] + paham["sub_queries"]

## 5. Turunan rencana — tanpa LLM

In [9]:
PEMERINGKAT = {"bernomor": "lexical", "perubahan": "lexical"}  # sisanya dense

SARINGAN_KOSONG = {"definisi-tabrakan", "multi-dokumen"}


def turunkan(paham):
    """Keluaran planner -> strategi retrieval. TIDAK menyentuh kalimat pengguna.

    Karena masukannya sudah terstruktur, fungsi ini bisa dites tanpa LLM sama
    sekali -- itu seluruh alasan keberadaannya.
    """
    tipe = paham["tipe"]
    mode = paham.get("temporal_mode") or "latest"
    kosong = tipe in SARINGAN_KOSONG
    return {
        "pemeringkat": PEMERINGKAT.get(tipe, "dense"),
        "saringan": {"peraturan": [], "pasal": []} if kosong else {
            "peraturan": paham.get("peraturan", []),
            "pasal": paham.get("pasal", []),
        },
        # Tidak ada flag cek_versi lagi: rantai SELALU diambil, tipe apa pun.
        # Daftar tipe yang dulu dilewati itu menebak di depan apa yang riwayat()
        # jawab dengan pasti di belakang -- pasal yang tak pernah diubah balik
        # daftar kosong, jadi ceknya gratis. Yang ditebak justru menerbitkan
        # bunyi usang tanpa satu pun tanda.
        "temporal": mode,
        # Berapa blok yang masuk BAHAN. Dulu 10 untuk semua pertanyaan: yang
        # sempit kebanyakan, yang luas justru kekurangan.
        "k": CAKUPAN.get(paham.get("cakupan"), CAKUPAN["sedang"]),
    }

#### cek: turunan rencana (5 kasus, tanpa LLM)

In [10]:
def _uji_turunan():
    """Soal asli dari ground_truth, dijalankan tanpa menyentuh LLM."""
    # "PP 55 tahun 2022" -- tahun di situ bagian NAMA, bukan titik waktu
    r = turunkan({"tipe": "bernomor", "peraturan": ["PP 55 tahun 2022"],
                  "pasal": ["57"], "temporal_mode": "latest"})
    assert r["pemeringkat"] == "lexical", r
    assert r["saringan"]["pasal"] == ["57"], r

    # flag cek_versi sudah dihapus -- rantai diambil untuk semua tipe
    assert "cek_versi" not in r, r

    # D01 "Dalam UU Penagihan Pajak ... siapa Pejabat?" -- dokumen disebut, versi tidak
    r = turunkan({"tipe": "definisi-otoritatif", "peraturan": ["UU Penagihan Pajak"],
                  "pasal": [], "temporal_mode": "latest"})
    assert r["pemeringkat"] == "dense", r

    # D07 "NITKU itu apa? Bedanya dengan NPWP?" -- saringan WAJIB kosong
    r = turunkan({"tipe": "definisi-tabrakan", "peraturan": ["PMK 81/2024"],
                  "pasal": [], "temporal_mode": "latest"})
    assert r["saringan"]["peraturan"] == [], "definisi-tabrakan tidak boleh disaring"

    # R04 "Bunyi Pasal 9 PMK 231/2019 setelah diubah PMK 59/2022" -- DUA peraturan.
    # Yang memuat bunyi barunya cuma 59/2022; kalau daftarnya menyusut jadi satu
    # di sini, saringan mengunci ke dokumen yang isinya justru bunyi lama.
    # DUA pasal dari DUA peraturan: dua-duanya harus lolos ke saringan
    r = turunkan({"tipe": "perubahan", "peraturan": ["PMK 231/2019", "PMK 59/2022"],
                  "pasal": ["9", "12"], "temporal_mode": "diff"})
    assert r["pemeringkat"] == "lexical", r
    assert r["saringan"]["peraturan"] == ["PMK 231/2019", "PMK 59/2022"], r
    assert r["saringan"]["pasal"] == ["9", "12"], "pasal kedua jangan hilang"

    # A01 "omzet 400 juta, bayar pajak ga?" -- tak ada peraturan disebut
    r = turunkan({"tipe": "tarif", "peraturan": [], "pasal": [], "temporal_mode": "latest"})
    assert r["saringan"]["peraturan"] == [], r

    # cakupan -> berapa blok. Yang tidak dikenal jatuh ke "sedang", bukan error:
    # planner meleset satu field tidak boleh mematikan seluruh pencarian.
    assert turunkan({"tipe": "bernomor", "cakupan": "sempit"})["k"] == 3
    assert turunkan({"tipe": "prosedur", "cakupan": "luas"})["k"] == 9
    assert turunkan({"tipe": "prosedur", "cakupan": "ngawur"})["k"] == 6
    assert turunkan({"tipe": "prosedur"})["k"] == 6, "field hilang -> sedang"

    print("ok  turunan rencana (tanpa LLM)")


_uji_turunan()

ok  turunan rencana (tanpa LLM)


## 6. Uji — satu pertanyaan

In [11]:
# -- model sel ini. Bawaannya ikut knop 1.2; timpa di sini kalau mau beda:
#    "" = lokal, "Groq" = Groq.
PAKAI_PLANNER = MODEL_PLANNER

PERTANYAAN = "Apa isi Pasal 57 PP 55 tahun 2022?"

paham = planner(PERTANYAAN, penyedia=PAKAI_PLANNER)

print(f"Q : {PERTANYAAN}   {paham['_detik']}s  ({paham['_model']})")
print()
print("-- dibaca model:")
print(json.dumps({k: v for k, v in paham.items()
                if not k.startswith("_") and k != "strategi"},
                indent=2, ensure_ascii=False))
print()
print(json.dumps(paham["strategi"], indent=2, ensure_ascii=False))

Q : Apa isi Pasal 57 PP 55 tahun 2022?   50.13s  (google/gemma-4-12b-qat)

-- dibaca model:
{
  "peraturan": [
    "PP 55 tahun 2022"
  ],
  "pasal": [
    "57"
  ],
  "temporal_mode": "latest",
  "tahun": "",
  "tipe": "bernomor",
  "cakupan": "sempit",
  "rewritten_query": "ketentuan yang diatur dalam Pasal 57 Peraturan Pemerintah Nomor 55 Tahun 2022",
  "sub_queries": [
    "Pasal 57 Peraturan Pemerintah Nomor 55 Tahun 2022",
    "ketentuan yang diatur dalam Pasal 57"
  ]
}

{
  "pemeringkat": "lexical",
  "saringan": {
    "peraturan": [
      "PP 55 tahun 2022"
    ],
    "pasal": [
      "57"
    ]
  },
  "temporal": "latest",
  "k": 3
}


## 7. Retrieval

### 7.1 Muat model embedding, reranker, dan Qdrant

In [12]:
# `embed.py` tinggal SATU, di akar proyek bareng parser & etl -- dulu ada
# salinan kedua di folder ini dan itu sempat menipu: perbaikan dipasang di
# akar, eval tetap memakai salinan lama yang menunjuk korpus PDF 139 dokumen.
import sys
from pathlib import Path as _P
sys.path.insert(0, str(_P.cwd().parent if _P.cwd().name == 'EVAL' else _P.cwd()))

import embed
from qdrant_client import QdrantClient, models
from sentence_transformers import CrossEncoder

VARIAN = "c"
PROBE = 50    # calon per kunci; RRF menggabung, reranker yang menyaring
K = 10        # yang akhirnya dipakai
RERANKER = "BAAI/bge-reranker-v2-m3"

qc = QdrantClient(embed.QDRANT, timeout=120)
model = embed.load_model()          # BGE-M3, ~1 menit pertama kali
COLL_UNIT = embed.coll(VARIAN)

# Reranker membaca pertanyaan dan teks BERSAMAAN, jadi jauh lebih teliti
# daripada mencocokkan dua vektor -- tapi juga jauh lebih lambat. Karena itu
# dia cuma dikasih calon hasil RRF, bukan 7.162 unit.
reranker = CrossEncoder(RERANKER, max_length=512, device="cuda")

print(COLL_UNIT, qc.count(COLL_UNIT).count, "unit")
print(embed.COLL_DOK, qc.count(embed.COLL_DOK).count, "dokumen")
print(f"probe {PROBE}/kunci -> rerank -> {K}")

d:\DDTC\Magang\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 393/393 [00:00<00:00, 1304.05it/s]


peraturan_unit_c 232081 unit
peraturan_dokumen 10693 dokumen
probe 50/kunci -> rerank -> 10


### 7.2 Kamus julukan peraturan
Vektor tidak pernah bisa bilang "tidak ketemu" -- 'UU KUP' dicocokkan lewat tabel dulu.

In [13]:
_JULUKAN = [
    (("kup", "ketentuan umum dan tata cara perpajakan"), ("uu", "6", "1983")),
    (("pph", "pajak penghasilan"), ("uu", "7", "1983")),
    (("ppn", "ppn dan ppnbm", "pajak pertambahan nilai"), ("uu", "8", "1983")),
    (("hpp", "harmonisasi peraturan perpajakan"), ("uu", "7", "2021")),
    (("ppsp", "penagihan pajak", "penagihan pajak dengan surat paksa"), ("uu", "19", "1997")),
    (("pbb", "pajak bumi dan bangunan"), ("uu", "12", "1985")),
    (("bm", "bea meterai", "bea materai"), ("uu", "10", "2020")),
    (("ta", "tax amnesty", "pengampunan pajak"), ("uu", "11", "2016")),
    (("pengadilan pajak",), ("uu", "14", "2002")),
    (("kepabeanan",), ("uu", "10", "1995")),
    (("ciptaker", "cipta kerja"), ("uu", "6", "2023")),
]
JULUKAN = {f"uu {a}": t for alias, t in _JULUKAN for a in alias}
# Jenis peraturan -> kode pendek. Dicocokkan dari yang PALING PANJANG duluan:
# "PERATURAN PEMERINTAH PENGGANTI UNDANG-UNDANG" berawalan "PERATURAN PEMERINTAH",
# jadi kalau urutannya terbalik satu-satunya Perpu di korpus tercatat sebagai PP.
# Dicocokkan setelah di-UPPER (lihat muat_tabel) -- `jenis_peraturan` dari kolom
# DB bentuknya Title Case, dari kop naskah HURUF BESAR. Diurut panjang duluan
# supaya "PERATURAN PEMERINTAH PENGGANTI UNDANG-UNDANG" tidak keburu ketangkap
# "PERATURAN PEMERINTAH".
JENIS = sorted([
    ("PERATURAN PEMERINTAH PENGGANTI UNDANG-UNDANG", "perpu"),
    ("PERATURAN DIREKTUR JENDERAL BEA DAN CUKAI", "perdjbc"),
    ("PERATURAN DIREKTUR JENDERAL PERBENDAHARAAN", "perdjpb"),
    ("PERATURAN MENTERI PERDAGANGAN", "permendag"),
    ("PERATURAN MENTERI DALAM NEGERI", "permendagri"),
    ("PERATURAN MENTERI KETENAGAKERJAAN", "permenaker"),
    ("PERATURAN MENTERI TENAGA KERJA", "permenaker"),
    ("PERATURAN MAHKAMAH AGUNG", "perma"),
    ("PERATURAN BANK INDONESIA", "pbi"),
    ("PERATURAN DIREKTUR JENDERAL PAJAK", "perdjp"),
    ("KEPUTUSAN DIREKTUR JENDERAL PAJAK", "kepdjp"),
    ("PERATURAN OTORITAS JASA KEUANGAN", "pojk"),
    ("PERATURAN MENTERI KEUANGAN", "pmk"),
    ("KEPUTUSAN MENTERI KEUANGAN", "kmk"),
    ("PERATURAN PRESIDEN", "perpres"),
    ("PERATURAN PEMERINTAH", "pp"),
    ("UNDANG-UNDANG", "uu"),
], key=lambda x: -len(x[0]))

# Sebutan yang ditulis orang -> kode yang sama. Urutan juga panjang duluan.
ALIAS = sorted([
    ("peraturan pemerintah pengganti undang undang", "perpu"), ("perppu", "perpu"), ("perpu", "perpu"),
    ("peraturan direktur jenderal bea dan cukai", "perdjbc"), ("perdirjen bc", "perdjbc"),
    ("per bc", "perdjbc"), ("djbc", "perdjbc"),
    ("peraturan direktur jenderal perbendaharaan", "perdjpb"), ("perdirjen perbendaharaan", "perdjpb"),
    ("peraturan menteri perdagangan", "permendag"), ("permendag", "permendag"),
    ("peraturan menteri dalam negeri", "permendagri"), ("permendagri", "permendagri"),
    ("peraturan menteri ketenagakerjaan", "permenaker"),
    ("peraturan menteri tenaga kerja", "permenaker"), ("permenaker", "permenaker"),
    ("peraturan mahkamah agung", "perma"), ("perma", "perma"),
    ("peraturan bank indonesia", "pbi"), ("pbi", "pbi"),
    ("peraturan otoritas jasa keuangan", "pojk"), ("pojk", "pojk"),
    ("peraturan menteri keuangan", "pmk"), ("pmk", "pmk"),
    ("keputusan menteri keuangan", "kmk"), ("kmk", "kmk"),
    ("peraturan presiden", "perpres"), ("perpres", "perpres"),
    ("peraturan pemerintah", "pp"), ("pp", "pp"),
    ("undang undang", "uu"), ("uu", "uu"),
], key=lambda x: -len(x[0]))


def _pokok(nomor):
    m = re.search(r"\d+", nomor or "")
    return str(int(m.group())) if m else None


def muat_tabel():
    """(kode jenis, nomor pokok, tahun) -> document_id, untuk SELURUH koleksi.

    Tiga hal yang berubah sejak korpus pindah ke Postgres, dan ketiganya diam:

    1. HURUF. `jenis_peraturan` dulu dibaca dari kop naskah -- HURUF BESAR SEMUA
       ("PERATURAN MENTERI KEUANGAN REPUBLIK INDONESIA"). Sekarang dari kolom DB
       dan bentuknya Title Case ("Peraturan Menteri Keuangan"), jadi
       `startswith` yang mematok huruf besar cocok NOL dari 6.580 dokumen --
       tabelnya kosong dan SEMUA sebutan bernomor jatuh ke pencarian vektor,
       yang tidak pernah bisa bilang "tidak ketemu".

    2. HALAMAN. `scroll(limit=1000)` cuma mengambil 1.000 titik pertama. Waktu
       korpusnya 139 dokumen itu cukup; di 6.580 dokumen, 85% tidak pernah masuk
       tabel. Sekarang ditelusuri sampai habis.

    3. TABRAKAN. 20 kunci menunjuk lebih dari satu dokumen -- nomor yang beda
       tapi angka pokoknya sama: 160/PMK.01/2008 vs 160.1/PMK.07/2008 vs
       160.3/PMK.07/2008. Dulu yang terakhir dibaca menang diam-diam. Sekarang
       kunci begitu DIBUANG: lebih baik jatuh ke vektor daripada menunjuk
       dokumen yang salah dengan yakin.
    """
    tabel, ganda, asing = {}, set(), set()
    titik, offset = [], None
    while True:
        batch, offset = qc.scroll(embed.COLL_DOK, limit=1000, offset=offset, with_payload=True)
        titik += batch
        if offset is None:
            break
    for t in titik:
        d = t.payload
        jenis = (d["jenis_peraturan"] or "").upper()
        kode = next((k for awalan, k in JENIS if jenis.startswith(awalan)), None)
        pokok = _pokok(d["nomor"])
        if kode is None or pokok is None:
            asing.add(d["jenis_peraturan"])
            continue
        kunci = (kode, pokok, d["tahun"])
        if kunci in tabel and tabel[kunci] != d["document_id"]:
            ganda.add(kunci)
        tabel[kunci] = d["document_id"]
    for k in ganda:
        del tabel[k]
    print(f"tabel dokumen: {len(tabel)} kunci dari {len(titik)} dokumen"
          f"  (ambigu dibuang: {len(ganda)} · jenis tak dikenal: {len(asing)})")
    return tabel, asing


TABEL, JENIS_ASING = muat_tabel()


def julukan(sebutan):
    s = re.sub(r"[^a-z0-9]+", " ", (sebutan or "").lower()).strip()
    kunci = JULUKAN.get(re.sub(r"^undang undang\b", "uu", s))
    return TABEL.get(kunci) if kunci else None



def baca_sebutan(sebutan):
    s = re.sub(r"[^a-z0-9]+", " ", (sebutan or "").lower()).strip()
    if not s:
        return None
    # PER-57/PJ/2010 dan KEP-220/PJ/2002: jenisnya menempel di nomor, bukan
    # ditulis terpisah. Penanda "pj" cukup -- tidak ada peraturan lain memakainya.
    if re.search(r"\bpj\b", s):
        kode = "kepdjp" if s.startswith("kep") else "perdjp"
    # Bea Cukai menomori "P-26/BC/2009" -- jenisnya menempel di nomor, persis
    # seperti /PJ/ di atas, dan awalannya cuma satu huruf ("P-") sehingga tidak
    # tertangkap daftar ALIAS mana pun. 411 dokumen di korpus.
    elif re.search(r"\bbc\b", s):
        kode = "perdjbc"
    else:
        kode = next((k for a, k in ALIAS if re.search(rf"\b{a}\b", s)), None)
    if kode is None:
        return None
    s = re.sub(r"\bpasal\s+\S+", " ", s)   # buang "Pasal 57" kalau ikut kebawa
    angka = re.findall(r"\d+", s)
    tahun = next((a for a in reversed(angka) if len(a) == 4 and a[:2] in ("19", "20")), None)
    nomor = next((a for a in angka if not (len(a) == 4 and a[:2] in ("19", "20"))), None)
    if not (tahun and nomor):
        return None
    return kode, str(int(nomor)), tahun


tabel dokumen: 9318 kunci dari 10693 dokumen  (ambigu dibuang: 221 · jenis tak dikenal: 62)


#### cek: julukan

In [14]:
assert julukan("UU KUP") == TABEL[("uu", "6", "1983")]
assert julukan("Undang-Undang KUP") == julukan("UU KUP")
assert julukan("pph") is None, "tanpa awalan uu harus meleset, bukan menebak"
assert julukan("PMK 81/2024") is None, "bukan julukan -- biar vektor yang urus"
# sebutan bernomor -> (kode, nomor, tahun). Murni teks, tanpa Qdrant.
assert baca_sebutan("PMK 81/2024") == ("pmk", "81", "2024")
assert baca_sebutan("PP 55 tahun 2022") == ("pp", "55", "2022")
assert baca_sebutan("PER-57/PJ/2010") == ("perdjp", "57", "2010")
assert baca_sebutan("KEP-220/PJ/2002") == ("kepdjp", "220", "2002")
assert baca_sebutan("10/PMK.03/2013") == ("pmk", "10", "2013"), baca_sebutan("10/PMK.03/2013")
assert baca_sebutan("Undang-Undang Nomor 6 Tahun 1983") == ("uu", "6", "1983")
assert baca_sebutan("Pasal 57 PP 55 tahun 2022") == ("pp", "55", "2022"), "nomor pasal jangan ikut terbaca"
assert baca_sebutan("PERPU 1/2013") == ("perpu", "1", "2013"), "Perpu bukan PP"
assert baca_sebutan("UU KUP") is None, "julukan bukan urusan sini -- biar kamus"
assert baca_sebutan("PMK 81") is None, "tanpa tahun jangan ditebak"

# tiap kunci harus benar-benar ada di tabel
for _s in ("PMK 81/2024", "PP 55 tahun 2022", "PER-57/PJ/2010", "UU 6 tahun 1983"):
    assert baca_sebutan(_s) in TABEL, (_s, baca_sebutan(_s))

menggantung = [a for a, k in JULUKAN.items() if k not in TABEL]
if JENIS_ASING:
    print(f"!! jenis tak dikenal, tidak masuk tabel: {sorted(JENIS_ASING)}")
print(f"{len(JULUKAN)} julukan, {len(TABEL)} dokumen di tabel"
    + (f" · !! menggantung: {menggantung}" if menggantung else ""))

!! jenis tak dikenal, tidak masuk tabel: ['Keputusan Bersama Direktur Jenderal', 'Keputusan Bersama Menteri', 'Keputusan Direktur Jenderal Bea dan Cukai', 'Keputusan Direktur Jenderal Perbendaharaan', 'Keputusan Direktur Jenderal Perdagangan Dalam Negeri', 'Keputusan Direktur Jenderal Perimbangan Keuangan', 'Keputusan Inspektur Jenderal', 'Keputusan Kepala Badan Kebijakan Fiskal', 'Keputusan Kepala Badan Pendidikan dan Pelatihan Keuangan', 'Keputusan Kepala Badan Pengawas Obat dan Makanan', 'Keputusan Kepala Pusat Pembinaan Profesi Keuangan', 'Keputusan Ketua Mahkamah Agung', 'Keputusan Ketua Pengadilan Pajak', 'Keputusan Menteri Investasi dan Hilirisasi/Kepala Badan Koordinasi Penanaman Modal', 'Keputusan Menteri Ketenagakerjaan', 'Keputusan Menteri Koordinator Bidang Politik, Hukum, dan Keamanan Republik Indonesia', 'Keputusan Menteri Pekerjaan Umum Dan Perumahan Rakyat', 'Keputusan Menteri Perdagangan', 'Keputusan Menteri Perindustrian', 'Keputusan Menteri Perindustrian Dan Perdagan

### 7.3 Pencarian, resolusi dokumen, dan tingkat saringan

In [15]:
def _vektor(teks):
    """-> (dense list, SparseVector). .lower() WAJIB: korpus diindeks lowercase."""
    d, s = embed.encode(model, [teks.lower()])
    return d[0].tolist(), s[0]


def _cari(coll, teks, using, limit, filt=None):
    """Satu kunci, satu collection. using: 'dense' (makna) atau 'sparse' (kata persis)."""
    d, s = _vektor(teks)
    return qc.query_points(coll, query=d if using == "dense" else s, using=using,
                        limit=limit, query_filter=filt, with_payload=True).points


def _lapor(sebutan, doc_id, jalur):
    """Cetak lalu kembalikan doc_id. SELALU dicetak: salah tebak dokumen tidak
    memunculkan error -- yang terjadi cuma hasil nol, dan sebabnya tak terlihat.

    Yang dicetak ID, bukan judul: ID inilah yang dipasang jadi saringan, jadi
    kalau hasilnya nol, ini yang perlu kamu cek langsung ke Qdrant. Nomor dan
    tahun sudah terbaca di dalam ID-nya sendiri.
    """
    print(f"  dokumen: {sebutan!r} -> {doc_id}   [{jalur}]")
    return doc_id


def cari_dokumen(sebutan):
    """'PP 55 tahun 2022' -> document_id asli. Tanpa ini saringan tidak bisa dipasang:
    ID punya hash yang tidak mungkin ditebak siapa pun."""
    if not sebutan:
        return None
    # Julukan diperiksa DULUAN. Vektor tidak pernah bisa bilang "tidak ketemu" --
    # dia selalu mengembalikan yang terdekat, sejauh apa pun; untuk "UU KUP"
    # kata "kup" bahkan nol kemunculan di seluruh 139 judul.
    if (dok := julukan(sebutan)):
        return _lapor(sebutan, dok, "kamus")
    # Sebutan bernomor: cocokkan PERSIS dulu. Kalau kuncinya tidak ada di tabel,
    # dokumennya memang tidak ada di korpus -- dan itu jawaban yang jujur, beda
    # dari vektor yang tetap menyodorkan judul terdekat.
    if (kunci := baca_sebutan(sebutan)) and kunci in TABEL:
        return _lapor(sebutan, TABEL[kunci], "tabel")
    d, s = _vektor(sebutan)
    h = qc.query_points(
        embed.COLL_DOK,
        prefetch=[models.Prefetch(query=d, using="dense", limit=10),
                models.Prefetch(query=s, using="sparse", limit=10)],
        query=models.FusionQuery(fusion=models.Fusion.RRF),
        limit=1, with_payload=True).points
    if not h:
        return None
    return _lapor(sebutan, h[0].payload["document_id"], "vektor")


def bikin_filter(doc_ids, pasal):
    """-> Filter Qdrant. Bagian yang kosong tidak dipasang.

    Beberapa dokumen digabung pakai MatchAny (ATAU), bukan beberapa kondisi
    must (DAN): satu unit cuma milik satu dokumen, jadi must dua dokumen
    selalu nol hasil.
    """
    must = []
    if doc_ids:
        must.append(models.FieldCondition(key="document_id",
                                        match=models.MatchAny(any=doc_ids)))
    if pasal:
        # planner keluarkan ["57"], payload menyimpan "Pasal 57".
        # ponytail: dokumen dan pasal disaring sebagai dua daftar terpisah, jadi
        # "Pasal 57 PP 55 dan Pasal 9 PMK 231" ikut memungut Pasal 9 PP 55 kalau
        # ada. Yang mahal itu kehilangan pasal kedua, bukan kelebihan blok --
        # reranker yang menyingkirkannya. Kalau nyasarnya kelihatan mengganggu,
        # naikkan ke pasangan (dokumen, pasal) pakai should[] berisi must[].
        # Dua bentuk label dirakit sekaligus: dokumen berbentuk penetapan
        # memakai `diktum KEEMPAT BELAS`, bukan `Pasal 14`. MatchAny itu ATAU,
        # jadi bentuk yang tidak terpakai cuma tidak pernah cocok.
        #
        # Tanpa ini "Diktum keempat belas KEP-238/PJ/2012" jatuh ke saringan
        # dokumen saja: 16 calon (= 16 diktumnya), lalu `cakupan sempit`
        # memotongnya jadi 3 dan unit yang DISEBUT PENANYA sendiri tidak ikut
        # terangkut. Alamat yang sudah disebut itu fakta, bukan bahan tebakan.
        label = ([f"Pasal {p}" for p in pasal]
                 + [f"diktum {p.upper()}" for p in pasal])
        must.append(models.FieldCondition(
            key="label", match=models.MatchAny(any=label)))
    return models.Filter(must=must) if must else None


def _uji_bikin_filter():
    """Label yang dirakit, bukan hasil pencarian -- tidak menyentuh Qdrant."""
    f = bikin_filter(["dok-a"], ["57", "keempat belas"])
    lab = next(k.match.any for k in f.must if k.key == "label")
    assert "Pasal 57" in lab, lab
    assert "diktum KEEMPAT BELAS" in lab, lab          # <- yang dulu hilang
    assert bikin_filter([], []) is None
    assert all(k.key != "label" for k in bikin_filter(["dok-a"], []).must)
    print("ok  bikin_filter (Pasal + diktum)")


_uji_bikin_filter()


def tingkat_saringan(doc_ids, pasal):
    """Saringan dari paling ketat ke paling longgar, tanpa tingkat kembar.

    Tingkat tengah bukan kasus pinggir: 85 dokumen memecah glosariumnya jadi
    unit per angka ("angka 64") dan NOL di antaranya punya unit berlabel
    "Pasal 1". Tiap pertanyaan "definisi X di Pasal 1 PMK Y" dijamin nol di
    tingkat pertama -- yang harus dilepas cuma pasalnya, dokumennya tidak
    salah apa-apa. Melepas dua-duanya sekaligus melebarkan pencarian dari satu
    dokumen ke 139.
    """
    urut = [("dokumen+pasal", bikin_filter(doc_ids, pasal)),
            ("dokumen saja", bikin_filter(doc_ids, "")),
            ("tanpa saringan", None)]
    keluar = []
    for nama, f in urut:
        if keluar and f == keluar[-1][1]:
            keluar[-1] = (nama, f)   # tingkat kembar: pakai nama yang lebih longgar
        else:
            keluar.append((nama, f))
    return keluar

ok  bikin_filter (Pasal + diktum)


#### cek: tingkat saringan tidak kembar, dan selalu ada jaring terakhir

In [16]:
# --- check: tidak ada tingkat kembar, dan selalu ada jaring terakhir.
assert [n for n, _ in tingkat_saringan(["a", "b"], ["9", "12"])] == \
    ["dokumen+pasal", "dokumen saja", "tanpa saringan"]
assert [n for n, _ in tingkat_saringan(["a"], [])] == ["dokumen saja", "tanpa saringan"]
assert [n for n, _ in tingkat_saringan([], ["9"])] == ["dokumen+pasal", "tanpa saringan"]
assert [n for n, _ in tingkat_saringan([], [])] == ["tanpa saringan"]

### 7.4 RRF, rerank, retrieve(), dan perakit konteks

In [17]:
def rrf(daftar, konstanta=60):
    """Gabung beberapa daftar peringkat. Pakai PERINGKAT, bukan skor: skor dense
    dan skor sparse beda skala, menjumlahkannya tidak berarti apa-apa."""
    skor, isi = {}, {}
    for hasil in daftar:
        for r, h in enumerate(hasil):
            skor[h.id] = skor.get(h.id, 0) + 1 / (konstanta + r + 1)
            isi[h.id] = h
    return [isi[i] for i in sorted(skor, key=skor.get, reverse=True)]


def rerank(query, hasil, k=K):
    """Urutkan ulang dengan CrossEncoder. Skornya ditempel ke tiap point."""
    if not hasil:
        return hasil
    skor = reranker.predict([(query, h.payload["teks"]) for h in hasil])
    for h, s in zip(hasil, skor):
        h.payload["_rerank"] = float(s)
    return sorted(hasil, key=lambda h: h.payload["_rerank"], reverse=True)[:k]


def retrieve(paham, pertanyaan, k=None, probe=PROBE, pakai_rerank=True):
    """Jalankan strategi planner. -> daftar point Qdrant, sudah diurutkan.

    k=None -> ikut cakupan yang dibaca planner. Sebut angkanya sendiri cuma
    kalau sedang membandingkan, bukan saat dipakai normal.
    """
    st = paham["strategi"]
    k = k or st.get("k", K)
    using = "sparse" if st["pemeringkat"] == "lexical" else "dense"
    doc_ids = [d for s in st["saringan"]["peraturan"] if (d := cari_dokumen(s))]
    kunci = kunci_pencarian(paham, pertanyaan)

    # Saringan terlalu ketat lebih berbahaya daripada terlalu longgar: hasilnya
    # nol, tanpa error. Turun SETINGKAT, bukan langsung lepas semuanya.
    for nama, filt in tingkat_saringan(doc_ids, st["saringan"]["pasal"]):
        hasil = rrf([_cari(COLL_UNIT, q, using, probe, filt) for q in kunci])
        if hasil:
            break
        # filt None = tingkat terakhir, tidak ada lagi yang bisa dilonggarkan
        print(f"  !! 0 hasil dengan saringan {nama}"
              + (" -> dilonggarkan" if filt is not None else ""))

    print(f"  {len(kunci)} kunci x {probe} -> {len(hasil)} calon unik"
          f"  [saringan: {nama}, ambil {k}]")
    # Daftar tingkat SELALU berujung di "tanpa saringan" dan dijalani sekali,
    # dari ketat ke longgar -- tidak pernah balik mengetat. Jadi nol di sini
    # berarti korpusnya yang tidak punya, bukan saringan yang kekencangan:
    # berhenti, mengulang cuma menghasilkan nol yang sama.
    if not hasil:
        print("  !! nol hasil bahkan tanpa saringan -- pencarian dihentikan")
        return hasil
    if not pakai_rerank:  # buat membandingkan: seberapa besar sumbangan reranker
        return hasil[:k]
    # ponytail: yang dikirim ke reranker rewritten_query, bukan kalimat asli --
    # korpusnya bahasa peraturan, dan rewritten_query sudah dalam bahasa itu.
    return rerank(paham["rewritten_query"] or pertanyaan, hasil, k)


def blok(h, n):
    """Satu unit, PERSIS seperti yang nanti dikirim ke LLM penjawab.

    Identitas dokumen dirakit balik di sini, bukan disimpan di tiap unit:
    payload memisahkannya supaya bisa disaring, tapi LLM butuh utuh -- tanpa
    nomor dan tahun, "Pasal 5" bisa milik 139 peraturan yang berbeda.
    """
    p = h.payload
    return (f"[{n}] {p['jenis_peraturan']} Nomor {p['nomor']} Tahun {p['tahun']}\n"
            f"    tentang {p['tentang']}\n"
            f"    {p['alamat'] or '-'}"
            + (" [PERUBAHAN]" if p.get("role") == "target" else "")
            + f"\n\n{p['teks']}")


PEMISAH = "\n\n" + "-" * 70 + "\n\n"


def konteks(hasil, n=None):
    """Bahan jawaban untuk LLM. Satu-satunya perakit -- tampil() memakai fungsi
    yang sama, jadi yang kamu baca di layar tidak bisa menyimpang dari yang
    dibaca model."""
    return PEMISAH.join(blok(h, i) for i, h in enumerate(hasil[:n or len(hasil)], 1))


def tampil(hasil, n=None):
    """konteks() + unit_id + skor rerank, untuk penelusuran.

    Tidak dipotong: yang paling perlu diperiksa justru bagian yang biasanya
    kepotong -- di situ letak ayat yang menentukan.
    """
    for i, h in enumerate(hasil[:n or len(hasil)], 1):
        p = h.payload
        skor = f"   rerank {p['_rerank']:+.3f}" if "_rerank" in p else ""
        print("\n" + "=" * 70)
        print(f"unit_id : {p['unit_id']}{skor}")
        print(blok(h, i))


print("siap: retrieve")

siap: retrieve


### Uji retrieval

In [18]:
# -- model sel ini (retrieval tidak pakai LLM; yang dipilih cuma planner-nya)
PAKAI_PLANNER = MODEL_PLANNER

PERTANYAAN = "Tenaga kerja asing dengan keahlian tertentu perlakuan PPh-nya bagaimana? Diatur di PP dan PMK yang mana?"

paham = planner(PERTANYAAN, penyedia=PAKAI_PLANNER)
print(f"Q : {PERTANYAAN}   {paham['_detik']}s  ({paham['_model']})")
print("strategi:", json.dumps(paham["strategi"], ensure_ascii=False))
print("kunci   :", kunci_pencarian(paham, PERTANYAAN))

hasil = retrieve(paham, PERTANYAAN)
print(f"\n{len(hasil)} hasil:")
tampil(hasil)

Q : Tenaga kerja asing dengan keahlian tertentu perlakuan PPh-nya bagaimana? Diatur di PP dan PMK yang mana?   37.55s  (google/gemma-4-12b-qat)
strategi: {"pemeringkat": "dense", "saringan": {"peraturan": [], "pasal": []}, "temporal": "latest", "k": 9}
kunci   : ['Tenaga kerja asing dengan keahlian tertentu perlakuan PPh-nya bagaimana? Diatur di PP dan PMK yang mana?', 'perlakuan Pajak Penghasilan (PPh) bagi Tenaga Kerja Asing dengan keahlian tertentu', 'dasar hukum Pajak Penghasilan (PPh) Tenaga Kerja Asing dalam Peraturan Pemerintah (PP)', 'dasar hukum Pajak Penghasilan (PPh) Tenaga Kerja Asing dalam Peraturan Menteri Keuangan (PMK)']
  4 kunci x 50 -> 126 calon unik  [saringan: tanpa saringan, ambil 9]

9 hasil:

unit_id : undang-undang-konsolidasi-pph-setelah-uu-cipta-kerja-db12680::bab-iii::pasal-4#b6   rerank +0.902
[1] Undang-Undang Nomor Konsolidasi PPh setelah UU Cipta Kerja Tahun 2020
    tentang 
    BAB III: OBJEK PAJAK > Pasal 4

Pasal 4
  (1a) Dikecualikan dari ketentuan 

## 8. Ekspansi versi

### 8.1 Naik ke akar, turun ke semua versi

In [19]:
ROMAWI = re.compile(r"Pasal [IVXLC]+$")


def _waktu(p):
    """Kunci urut satu versi.

    ponytail: tanggal_berlaku (terisi 138/139 dokumen) jatuh ke `tahun` kalau
    kosong. Yang kosong cuma UU 7/1994, dan itu benar: berlakunya digantungkan
    ke Persetujuan WTO, bukan tanggal. Dua-duanya ISO ("2011-06-06" / "2011"),
    jadi perbandingan string sudah urut benar tanpa parsing tanggal.
    """
    return p.get("tanggal_berlaku") or str(p.get("tahun") or "")


def _di(doc_id, label, limit=256):
    """Unit berlabel `label` DI dalam dokumen `doc_id`.

    scroll, bukan query_points: tidak ada vektor, tidak ada model, tidak ada
    peringkat. Itu sebabnya cek versi praktis gratis.

    Batasnya dinaikkan dari 64 waktu korpus pindah ke Postgres. Diukur ke
    206.946 unit: satu pasal terpanjang punya 65 bagian (Permen PANRB 66/2021
    Pasal 8), jadi batas lama memotongnya DIAM-DIAM -- rantai versinya jadi
    kurang satu tanpa ada yang bersuara. 256 menyisakan ruang, dan scroll
    berfilter di field ber-index murah.
    """
    return qc.scroll(COLL_UNIT, limit=limit, with_payload=True,
                     scroll_filter=models.Filter(must=[
                         models.FieldCondition(key="document_id",
                                               match=models.MatchValue(value=doc_id)),
                         models.FieldCondition(key="label",
                                               match=models.MatchValue(value=label))]))[0]


def _pengganti(doc_id, label, limit=256):
    """Unit yang MENGGANTI (doc_id, label) -- arah kebalikan dari _di()."""
    return qc.scroll(COLL_UNIT, limit=limit, with_payload=True,
                     scroll_filter=models.Filter(must=[
                         models.FieldCondition(key="target_document_id",
                                               match=models.MatchValue(value=doc_id)),
                         models.FieldCondition(key="label",
                                               match=models.MatchValue(value=label))]))[0]


def akar(doc_id, label, batas=8):
    """Naik ke versi paling asal. -> (doc_id, label, sebab berhenti).

    Nomor pasal dibawa terus sepanjang naik: PMK 53/2025 menulis "Pasal 313",
    yang diubahnya juga "Pasal 313", akarnya juga. Yang berganti cuma
    dokumennya -- itu sebabnya satu label cukup jadi pegangan.

    Empat sebab berhenti, dan bedanya penting:
      asli        ketemu unit standalone -- ini teks aslinya
      luar-korpus yang diubah belum ada di 139 dokumen. BUKAN "tidak pernah
                  berubah", tapi "tidak tahu" (581 unit menunjuk ke luar)
      sisipan     pasalnya tidak ada di dokumen target: pasal baru yang
                  disisipkan ("Pasal 3B", "6A", "1A"). Rantainya mulai dari
                  tengah, dan itu memang benar -- 116 kasus
      putaran     data melingkar; berhenti daripada menggantung
    """
    jejak = set()
    for _ in range(batas):
        if (doc_id, label) in jejak:
            return doc_id, label, "putaran"
        jejak.add((doc_id, label))
        di_sini = _di(doc_id, label)
        if not di_sini:
            return doc_id, label, "sisipan"
        # Standalone menang kalau satu dokumen punya dua-duanya (1,6% label
        # ambigu): berhenti di tempat lebih aman daripada naik ke silsilah lain.
        if any(u.payload.get("role") != "target" for u in di_sini):
            return doc_id, label, "asli"
        naik = next((u.payload["target_document_id"] for u in di_sini
                     if u.payload.get("target_document_id")), None)
        if naik is None:
            return doc_id, label, "luar-korpus"
        doc_id = naik
    return doc_id, label, "terlalu dalam"


def gabung(unit):
    """Pecahan `bagian` -> satu teks utuh.

    Tiap pecahan MENGULANG kepala yang sama supaya bisa dibaca berdiri sendiri:
    preamble ("Ketentuan Pasal 1 diubah, sehingga ..."), label pasalnya, dan
    sering batang ayatnya juga ("(1) Pemungut pajak adalah:"). Disambung mentah,
    kepala itu muncul berkali-kali -- dan yang mahal bukan panjangnya, melainkan
    LLM membaca SATU daftar yang terpotong sebagai beberapa aturan berbeda.

    Dibandingkan PER POTONGAN terhadap potongan pertama, bukan awalan bersama
    seluruh potongan: satu potongan yang menyimpang sudah cukup membatalkan
    awalan bersama -- dan potongan terakhir memang sering mulai di ayat lain
    ("(1a)"), persis kasus yang membuat baris batang ayat lolos tiga kali.

    Yang dibuang cuma baris yang SAMA PERSIS dan berurutan dari atas. Baris
    kembar yang sah di tengah teks ("dihapus.") tidak tersentuh.
    """
    unit = sorted(unit, key=lambda u: int(str(u.payload.get("bagian") or "1/1").split("/")[0]))
    teks = [u.payload["teks"] for u in unit]
    if len(teks) == 1:
        return teks[0]
    kepala = teks[0].split("\n")
    keluar = [teks[0]]
    for t in teks[1:]:
        baris = t.split("\n")
        n = 0
        while n < len(baris) and n < len(kepala) and baris[n] == kepala[n]:
            n += 1
        keluar.append("\n".join(baris[n:]))
    return "\n".join(x for x in keluar if x.strip())


print("siap: akar, gabung")


siap: akar, gabung


### 8.2 Pengantar rantai -- keterangan riwayat untuk LLM

In [20]:
CATATAN = {
    "sisipan": "pasal SISIPAN -- pasal baru yang ditambahkan, tidak ada teks asli",
    "luar-korpus": "peraturan yang diubah TIDAK ADA di korpus -- riwayat TIDAK LENGKAP, "
                   "jangan simpulkan 'tidak pernah berubah'",
    "diganti": "peraturan ini SUDAH DIGANTI seluruhnya oleh peraturan lain -- "
               "pasalnya memang tidak pernah diubah satu per satu, tapi JANGAN "
               "disimpulkan masih berlaku; sebutkan penggantinya",
    "putaran": "data melingkar -- riwayat tidak bisa dipercaya penuh",
    "terlalu dalam": "rantai lebih dalam dari batas -- riwayat mungkin terpotong",
    "sebelum-versi-awal": "tahun yang ditanya lebih awal dari versi paling awal yang ada -- "
                          "yang ditampilkan versi PERTAMA, bukan yang berlaku saat itu",
    "tanggal-tidak-lengkap": "tanggal berlaku tidak lengkap -- versi yang berlaku pada tahun "
                             "itu tidak bisa dipastikan, dua versi terakhir ditampilkan",
}


def pilih_versi(semua, mode, tahun=""):
    """Rantai versi + maksud waktu -> versi yang BENAR-BENAR dikirim.

    -> (daftar versi, catatan). Murni perbandingan tanggal: tidak menyentuh
    Qdrant maupun LLM, jadi bisa diuji apa adanya.

    Pada satu tanggal cuma ada SATU bunyi yang berlaku -- itu sebabnya "as_of"
    mengirim satu versi, bukan mengirim semuanya lalu menyuruh penjawab
    memilih. Memilih berdasarkan tanggal justru pekerjaan yang paling tidak
    bisa diandalkan kalau diserahkan ke model, dan datanya sudah ada di sini.
    """
    if not semua:
        return [], ""
    if mode == "all":
        return semua, ""
    if mode == "riwayat":       # daftar perubahannya sudah ada di pengantar
        return [], ""
    if mode == "diff":
        return semua[-2:], ""
    if mode == "as_of" and (m := re.search(r"(19|20)\d{2}", tahun or "")):
        thn = m.group()          # "1 Januari 2013" / "2013-05-01" -> "2013".
        # Dibanding apa adanya, "2013" <= "1 Januari 2013" itu False dan seluruh
        # rantai dinyatakan terlalu muda -- salah, dengan catatan yang menyesatkan.
        if any(not w for w, _ in semua):
            return semua[-2:], "tanggal-tidak-lengkap"
        berlaku = [v for v in semua if v[0][:4] <= thn]
        if not berlaku:
            return semua[:1], "sebelum-versi-awal"
        return berlaku[-1:], ""
    return semua[-1:], ""       # latest, dan mode apa pun yang tidak dikenal


def riwayat(doc_id, label):
    """Semua versi (doc_id, label), urut waktu. -> (daftar versi, sebab).

    Satu lookup dari akar sudah cukup -- tidak perlu meloncat satu per satu:
    pengubah menunjuk AKAR, bukan pengubah sebelumnya. PMK 11/2025 dan PMK
    53/2025 dua-duanya menunjuk PMK 81/2024, jadi sekali turun dari akar
    rantainya sudah utuh.

    Daftar kosong BELUM TENTU "tidak pernah diubah". Peraturan yang DICABUT
    UTUH tidak meninggalkan jejak apa pun di sini: penggantinya tidak mengutip
    satu pasal pun, dia menulis ulang dari nol, jadi tak ada unit yang
    `target_document_id`-nya menunjuk ke peraturan lama. 2.355 dokumen di korpus
    begitu -- dan untuk semuanya jawaban lama "tidak pernah diubah" itu justru
    kebalikan dari kenyataan.

    Yang menangkapnya `penerus` di payload, dari `peraturan_relasi` (lihat
    sel 10 etl.py). Karena itu daftar kosong dipisah jadi dua sebab:
      diganti     ada penerus -> peraturannya sudah mati, sebutkan penggantinya
      <sebab akar> tidak ada penerus -> memang tidak pernah diubah
    """
    root, lab, sebab = akar(doc_id, label)
    ganti = [u for u in _pengganti(root, lab)
             if not ROMAWI.match(u.payload.get("label") or "")]
    if not ganti:
        # cek dulu apakah peraturannya punya penerus sebelum menyimpulkan apa pun
        di_akar = _di(root, lab)
        if any(u.payload.get("penerus") for u in di_akar):
            return [], "diganti"
        return [], sebab

    per_dok = {}
    for u in ganti:
        per_dok.setdefault(u.payload["document_id"], []).append(u)

    versi = []
    if sebab == "asli":
        asli = [u for u in _di(root, lab) if u.payload.get("role") != "target"]
        if asli:
            versi.append((_waktu(asli[0].payload), asli))
    versi += [(_waktu(us[0].payload), us) for us in per_dok.values()]
    versi.sort(key=lambda v: v[0])
    return versi, sebab


def ekspansi(hasil, paham, maks_pasal=4):
    """Cek versi atas hasil pencarian. -> daftar rantai, satu per pasal.

    Kunci dikumpulkan dulu baru ditanya: satu pasal sering pecah jadi 6 unit,
    jadi 10 hasil bisa cuma 3 pasal berbeda -- bertanya per hit membuang 7
    kueri untuk jawaban yang sama.

    `semua` disimpan terpisah dari `versi`: mode latest cuma MENAMPILKAN bunyi
    terakhir, tapi tetap harus MEMBERI TAHU berapa kali diubah dan oleh siapa --
    tanpa itu LLM menulis dasar hukum "PER-31/PJ/2015" padahal yang benar
    "Pasal 1 PER-57/PJ/2010 sebagaimana telah diubah dengan PER-31/PJ/2015".

    Rantai dipasang SEKALIPUN bunyinya sudah ikut terjaring pencarian. Dulu
    dilewati kalau sudah ada, dan itu justru membuang keterangannya: yang
    tersisa satu blok berkepala "PER-31/PJ/2015" tanpa jejak bahwa dia Pasal 1
    milik PER-57/PJ/2010. Makin bagus pencariannya, makin sering keterangan itu
    hilang. Teks yang dobel bukan urusan di sini -- rapikan() membuangnya,
    dengan membandingkan unit_id hasil terhadap unit_id rantai.
    """
    mode = paham["strategi"]["temporal"]
    dilihat, keluar = set(), []

    for h in hasil:
        p = h.payload
        lab = p.get("label") or ""
        if not lab or ROMAWI.match(lab):
            continue
        # Kalau hit ini sendiri bunyi baru (role=target), pijakannya AKAR-nya,
        # bukan dokumennya sendiri: tidak ada yang mengubah peraturan pengubah,
        # jadi mencari dari situ dijamin nol.
        kunci = (p.get("target_document_id") or p["document_id"], lab)
        if kunci in dilihat:
            continue
        dilihat.add(kunci)

        versi, sebab = riwayat(*kunci)
        if versi:
            pilih, nota = pilih_versi(versi, mode, paham.get("tahun", ""))
            keluar.append({"pasal": lab, "sebab": sebab, "akar": kunci[0],
                           "versi": pilih, "semua": versi, "mode": mode,
                           "tahun": paham.get("tahun", ""), "catatan": nota})
        elif sebab == "diganti":
            # Rantai kosong TAPI peraturannya sudah punya pengganti. Dulu kasus
            # ini ikut terbuang bersama "tidak pernah diubah", dan itu justru
            # kebalikan dari kenyataan -- 2.355 dokumen di korpus begitu.
            # Teksnya tidak ada yang perlu dikirim; yang perlu cuma
            # keterangannya, supaya penjawab tidak menyimpulkan masih berlaku.
            pen = next((u.payload["penerus"] for u in _di(*kunci)
                        if u.payload.get("penerus")), [])
            keluar.append({"pasal": lab, "sebab": sebab, "akar": kunci[0],
                           "versi": [], "semua": [], "mode": mode,
                           "tahun": paham.get("tahun", ""), "catatan": None,
                           "penerus": pen})
        if len(dilihat) >= maks_pasal:
            break
    return keluar


def _dok(doc_id):
    """Identitas dokumen dari koleksi dokumen. Dipakai kalau akarnya tidak punya
    unit sendiri (pasal sisipan) -- namanya tetap harus ikut disebut."""
    t = qc.scroll(embed.COLL_DOK, limit=1, with_payload=True,
                  scroll_filter=models.Filter(must=[models.FieldCondition(
                      key="document_id", match=models.MatchValue(value=doc_id))]))[0]
    return t[0].payload if t else None


def _sebut(p):
    """Sebutan resmi satu dokumen. Dipakai di DASAR HUKUM yang keluar ke pengguna,
    jadi bentuknya harus benar.

    "Tahun {tahun}" cuma ditempel kalau nomornya belum memuat tahun. Sejak
    korpus pindah ke Postgres, `nomor` diambil apa adanya dari kolom DB dan
    hampir selalu SUDAH memuat tahunnya:
        "1 Tahun 2019"      -> "Nomor 1 Tahun 2019 Tahun 2019"   (kembar, 2.293 dok)
        "234/PMK.011/2008"  -> "Nomor 234/PMK.011/2008 Tahun 2008" (mubazir, 4.276 dok)
    Dua-duanya salah kutip, dan penjawab menyalinnya apa adanya sebagai dasar
    hukum.
    """
    if not p:
        return "(?)"
    nomor, tahun = str(p.get("nomor") or ""), str(p.get("tahun") or "")
    ekor = "" if (tahun and tahun in nomor) else f" Tahun {tahun}"
    return f"{p['jenis_peraturan']} Nomor {nomor}{ekor}".strip()


def _pengantar(r):
    """Kalimat pembuka satu rantai: FAKTA saja -- dokumen akarnya apa, berapa
    kali diubah, oleh siapa, mana yang berlaku, dan apa yang tidak kita tahu.

    Cara MENJAWAB tidak ditulis di sini; itu tugas system prompt penjawab.
    Yang di sini cuma hal yang tidak bisa disimpulkan dari blok teks di bawahnya.
    """
    pokok = _dok(r["akar"])
    ganti = [(w, us[0].payload) for w, us in r["semua"]
             if us[0].payload.get("role") == "target"]
    baris = [f"### {r['pasal']} — {_sebut(pokok)}"]

    if ganti:
        # Versi yang TIDAK dikirim tetap disebut di sini -- namanya dan tanggalnya
        # saja, beberapa baris. Penjawab jadi tahu ada versi lain tanpa harus
        # dikirimi teksnya, dan tidak bisa bilang "tidak pernah diubah lagi".
        dipilih = {w for w, _ in r["versi"]}
        baris.append(f"### diubah {len(ganti)} kali:")
        baris += [f"###   {w}  {_sebut(p)}"
                  + ("   <- INI yang ditampilkan di bawah" if w in dipilih else "")
                  for w, p in ganti]
        # Rumus penyebutan mengikuti versi yang DITAMPILKAN. Kalau yang dikirim
        # bunyi 2013, menyebut "terakhir dengan PER-31/PJ/2015" itu salah zaman:
        # penjawab akan menyalin kalimat ini apa adanya sebagai dasar hukum.
        batas = max(dipilih) if (r["mode"] == "as_of" and dipilih) else ganti[-1][0]
        sampai = [g for g in ganti if g[0] <= batas] or ganti
        akhir = _sebut(sampai[-1][1])
        rumus = (f"sebagaimana telah diubah dengan {akhir}" if len(sampai) == 1 else
                 f"sebagaimana telah beberapa kali diubah, terakhir dengan {akhir}")
        kapan = f" pada {r['tahun']}" if r["mode"] == "as_of" and r.get("tahun") else ""
        baris.append(f"### bunyi yang berlaku{kapan}: {r['pasal']} {_sebut(pokok)} {rumus}")

    # Nama penggantinya ikut disebut -- "sudah diganti" tanpa menyebut oleh apa
    # tidak bisa dipakai penjawab untuk menunjukkan aturan yang benar.
    for did in r.get("penerus") or []:
        baris.append(f"###   diganti oleh: {_sebut(_dok(did))}")

    if not r["versi"]:
        # tidak ada teks yang dikirim -- baris "ditampilkan" cuma menyesatkan
        for k in (r["sebab"], r.get("catatan")):
            if k in CATATAN:
                baris.append(f"### !! {CATATAN[k]}")
        return "\n".join(baris)

    baris.append("### ditampilkan: " + {
        "all": "SEMUA versi, urut waktu -- yang PALING BAWAH yang berlaku",
        "riwayat": "DAFTAR PERUBAHAN saja -- bunyi pasalnya sengaja tidak dikirim",
        "diff": "DUA versi: sebelum dan sesudah perubahan yang ditanyakan",
        "as_of": f"bunyi yang BERLAKU pada {r.get('tahun') or '-'}",
    }.get(r["mode"], "bunyi TERAKHIR saja"))
    for k in (r["sebab"], r.get("catatan")):
        if k in CATATAN:
            baris.append(f"### !! {CATATAN[k]}")
    return "\n".join(baris)


def blok_versi(rantai, mulai=1):
    """Rantai versi -> teks, format sama dengan blok().

    Satu perakit saja, seperti konteks(): yang kamu baca di layar tidak boleh
    menyimpang dari yang dibaca model.
    """
    potong, n = [], mulai
    for r in rantai:
        potong.append(_pengantar(r))
        akhir = len(r["versi"]) - 1
        for i, (waktu, unit) in enumerate(r["versi"]):
            p = unit[0].payload
            tag = (f"BERLAKU PADA {r.get('tahun') or '?'}" if r["mode"] == "as_of" else
                   ("SEBELUM" if i == 0 else "SESUDAH") if r["mode"] == "diff" and akhir else
                   "BERLAKU" if i == akhir else
                   "ASLI" if p.get("role") != "target" else f"PERUBAHAN {i}")
            potong.append(
                f"[{n}] [{tag}] berlaku {waktu}\n"
                f"    {_sebut(p)}\n"
                f"    tentang {p['tentang']}\n"
                f"    {p['alamat'] or '-'}\n\n{gabung(unit)}")
            n += 1
    return PEMISAH.join(potong)


print("siap: riwayat, ekspansi, blok_versi")


siap: riwayat, ekspansi, blok_versi


### 8.3 _id_dok() -- document_id dari (nomor, tahun), tanpa vektor

In [21]:
def _id_dok(nomor, tahun):
    """document_id dari (nomor, tahun) apa adanya -- tanpa vektor, supaya uji di
    bawah tidak ikut gagal gara-gara pencarian dokumen meleset."""
    t = qc.scroll(embed.COLL_DOK, limit=1, with_payload=True,
                scroll_filter=models.Filter(must=[
                    models.FieldCondition(key="nomor", match=models.MatchValue(value=nomor)),
                    models.FieldCondition(key="tahun", match=models.MatchValue(value=tahun))]))[0]
    assert t, f"dokumen {nomor}/{tahun} tidak ada di {embed.COLL_DOK}"
    return t[0].payload["document_id"]

#### cek: ekspansi versi (6 kasus, lawan indeks)

In [22]:
# --- cek: pilih_versi. Murni tanggal -- tidak menyentuh Qdrant sama sekali.
_v = [("2010-01-01", "asli"), ("2013-04-01", "r1"), ("2015-08-01", "r2")]
assert pilih_versi(_v, "latest") == (_v[-1:], "")
assert pilih_versi(_v, "all") == (_v, "")
assert pilih_versi(_v, "riwayat") == ([], ""), "riwayat tidak mengirim teks pasal"
assert pilih_versi(_v, "diff") == (_v[-2:], "")
assert pilih_versi(_v, "as_of", "2014") == (_v[1:2], ""), "2014 -> revisi 2013"
assert pilih_versi(_v, "as_of", "2013") == (_v[1:2], ""), "tahun sama -> versi tahun itu"
assert pilih_versi(_v, "as_of", "2026") == (_v[-1:], "")
assert pilih_versi(_v, "as_of", "2009") == (_v[:1], "sebelum-versi-awal")
assert pilih_versi(_v, "as_of", "") == (_v[-1:], ""), "as_of tanpa tahun -> jangan menebak"
assert pilih_versi(_v, "as_of", "1 Januari 2014") == (_v[1:2], ""), "tahun ditulis panjang"
assert pilih_versi(_v, "as_of", "2014-06-30") == (_v[1:2], ""), "tahun ditulis sebagai tanggal"
assert pilih_versi(_v, "as_of", "tahun lalu") == (_v[-1:], ""), "tanpa angka tahun -> latest"
assert pilih_versi(_v, "ngawur") == (_v[-1:], ""), "mode asing -> latest, bukan error"
assert pilih_versi([("", "x"), ("2015", "y")], "as_of", "2014")[1] == "tanggal-tidak-lengkap"
assert pilih_versi([], "all") == ([], "")
print("ok  pilih_versi (15 kasus, tanpa indeks)")


def _uji_ekspansi():
    """Semua angka di sini sudah diverifikasi langsung di unit/*.units.json.
    Kalau uji ini gagal, yang salah indeksnya -- bukan datanya."""
    p57 = _id_dok("PER-57/PJ/2010", "2010")

    # Pasal 1: asli 2010, diubah PER-06/2013 dan PER-31/2015
    versi, sebab = riwayat(p57, "Pasal 1")
    assert sebab == "asli", sebab
    assert len(versi) >= 3, [w for w, _ in versi]
    assert [w for w, _ in versi] == sorted(w for w, _ in versi), "harus urut waktu"
    assert versi[0][1][0].payload["document_id"] == p57, "yang paling awal harus aslinya"

    # Pasal 3B tidak pernah ada di PER-57/PJ/2010 (dokumennya cuma Pasal 1-8)
    versi, sebab = riwayat(p57, "Pasal 3B")
    assert sebab == "sisipan", sebab
    assert versi, "sisipan tetap punya rantai, cuma tanpa [ASLI]"

    # Multi-hop: PMK 53/2025 -> PMK 11/2025 -> PMK 81/2024.
    # Nomornya ditulis lengkap ("53 Tahun 2025") karena sejak korpus pindah ke
    # Postgres, `nomor` diambil apa adanya dari kolom DB -- dulu hasil bacaan
    # teks yang cuma menyisakan angkanya ("53").
    root, lab, _ = akar(_id_dok("53 Tahun 2025", "2025"), "Pasal 313")
    assert root == _id_dok("81 Tahun 2024", "2024"), root
    assert lab == "Pasal 313"

    # Pecahan digabung tanpa mengulang preamble. PER-06/2013 memecah Pasal 3B
    # jadi 6 unit, dan keenamnya mengulang kalimat pembuka yang sama.
    p06 = _id_dok("PER-06/PJ/2013", "2013")
    unit = [u for u in _pengganti(p57, "Pasal 3B")
            if u.payload["document_id"] == p06]
    assert len(unit) > 1, "harusnya terpecah beberapa bagian"
    assert gabung(unit).count("berbunyi sebagai berikut") == 1, "preamble terulang"

    # Kepala yang terulang BUKAN cuma preamble: potongan juga mengulang batang
    # ayatnya. Kalau lolos, LLM membaca satu daftar terpotong sebagai beberapa
    # aturan berbeda -- 54 pasal di korpus ini kena.
    p31 = _id_dok("PER-31/PJ/2015", "2015")
    satu = [u for u in _pengganti(p57, "Pasal 1") if u.payload["document_id"] == p31]
    assert len(satu) == 4, len(satu)
    assert gabung(satu).count("(1) Pemungut pajak sebagaimana dimaksud") == 1, \
        "batang ayat terulang -- daftar terbaca sebagai beberapa aturan"

    # Pengantar rantai: mode latest pun WAJIB menyebut akar & jumlah perubahan
    versi, sebab = riwayat(p57, "Pasal 1")
    r = {"pasal": "Pasal 1", "sebab": sebab, "akar": p57,
        "versi": versi[-1:], "semua": versi, "mode": "latest"}
    teks = _pengantar(r)
    assert "PER-57/PJ/2010" in teks, teks
    assert "diubah" in teks and "kali" in teks, teks
    assert "bunyi TERAKHIR" in teks, teks

    # Bunyi terbaru sudah ada di "hasil pencarian" -> rantainya TETAP dipasang.
    # Yang dibawa rantai bukan cuma teks, tapi keterangan riwayatnya.
    terbaru = riwayat(p57, "Pasal 1")[0][-1][1]
    r_ada = ekspansi(terbaru, {"strategi": {"temporal": "latest"},
                               "tahun": ""})
    assert r_ada, "rantai hilang gara-gara bunyinya sudah terjaring pencarian"
    assert "diubah" in _pengantar(r_ada[0]), "keterangan riwayat ikut hilang"

    print("ok  ekspansi versi (10 kasus, lawan indeks)")
    print()
    print("-- contoh pengantar (mode latest):")
    print(teks)


_uji_ekspansi()

ok  pilih_versi (15 kasus, tanpa indeks)
ok  ekspansi versi (10 kasus, lawan indeks)

-- contoh pengantar (mode latest):
### Pasal 1 — Peraturan Direktur Jenderal Pajak Nomor PER-57/PJ/2010
### diubah 2 kali:
###   2013-03-07  Peraturan Direktur Jenderal Pajak Nomor PER-06/PJ/2013
###   2015-08-05  Peraturan Direktur Jenderal Pajak Nomor PER-31/PJ/2015   <- INI yang ditampilkan di bawah
### bunyi yang berlaku: Pasal 1 Peraturan Direktur Jenderal Pajak Nomor PER-57/PJ/2010 sebagaimana telah beberapa kali diubah, terakhir dengan Peraturan Direktur Jenderal Pajak Nomor PER-31/PJ/2015
### ditampilkan: bunyi TERAKHIR saja


### Dedupe & rakit ulang

In [23]:
def rapikan(hasil, rantai=(), rakit=True):
    """Buang yang dobel sebelum masuk LLM. -> daftar point, urutan dipertahankan.

    Tiga sumber dobel, ketiganya terukur di korpus ini:

      1. pecahan `bagian` -- 866 pasal terpecah jadi 2.522 unit (35% korpus).
         Sepuluh hasil pencarian bisa cuma 3 pasal, dengan kalimat pembuka yang
         sama diulang sepuluh kali.
      2. unit yang sudah ikut di rantai ekspansi -- kalau tidak dibuang, teks
         yang sama muncul dua kali dengan tanda yang berbeda.
      3. pasal yang BOLONG: kalau cuma potongan 2 dan 5 yang terambil, yang
         dibaca LLM pasal berlubang tanpa ada tandanya.
    """
    dipakai = {u.payload["unit_id"] for r in rantai for _, us in r["versi"] for u in us}
    keluar, sudah = [], set()
    for h in hasil:
        p = h.payload
        if p["unit_id"] in dipakai:
            continue
        kunci = (p["document_id"], p.get("alamat"))
        if kunci in sudah:
            continue
        sudah.add(kunci)
        if rakit and p.get("bagian"):
            # ponytail: pasal dirakit UTUH, bukan sepotong yang kebetulan
            # terambil. Batasnya: pasal 7 potongan jadi satu blok panjang --
            # kalau konteks membengkak, turunkan K, JANGAN memotong pasalnya.
            # batas dilepas ke bawaan _di (256): ada 8 pasal berbagian lebih
            # dari 32, dan kalau terpotong pasalnya dirakit TIDAK utuh --
            # persis yang mau dicegah blok ini.
            saudara = [u for u in _di(p["document_id"], p["label"])
                       if u.payload.get("alamat") == p.get("alamat")]
            if len(saudara) > 1:
                p["teks"] = gabung(saudara)
                p["bagian"] = f"utuh ({len(saudara)} bagian)"
        keluar.append(h)
    return keluar

#### cek: rapikan (4 kasus)

In [24]:
def _uji_rapikan():
    p57 = _id_dok("PER-57/PJ/2010", "2010")
    p06 = _id_dok("PER-06/PJ/2013", "2013")
    enam = [u for u in _pengganti(p57, "Pasal 3B")
            if u.payload["document_id"] == p06]
    assert len(enam) == 6, len(enam)
    utuh = gabung(enam)                      # dihitung DULU: rapikan menulis ke payload

    dua = rapikan(enam[:2])                  # dua potongan -> satu blok
    assert len(dua) == 1, len(dua)
    assert dua[0].payload["teks"] == utuh, "harus dirakit UTUH, bukan 2 potong"
    assert utuh.count("berbunyi sebagai berikut") == 1, "kalimat pembuka terulang"

    # unit yang sudah dipakai rantai ekspansi tidak boleh muncul lagi
    versi, sebab = riwayat(p57, "Pasal 3B")
    rantai = [{"pasal": "Pasal 3B", "sebab": sebab, "akar": p57,
               "versi": versi, "semua": versi, "mode": "all"}]
    assert rapikan(enam[:2], rantai) == [], "sudah ada di rantai, harusnya dibuang"

    print("ok  rapikan (4 kasus)")


_uji_rapikan()

ok  rapikan (4 kasus)


## 9. Penjawab

### 9.1 System prompt penjawab

In [25]:
SISTEM_JAWAB = """# PERAN

Kamu asisten hukum pajak Indonesia. Yang bertanya bisa siapa saja konsultan
yang sedang menyiapkan pendapat, staf pajak yang mengurus kewajiban kantornya,
atau orang awam yang bingung. Semuanya butuh jawaban yang bisa dipakai, bukan
rangkuman kabur.

Kamu menerima BAHAN berupa kutipan peraturan yang sudah dicarikan sistem, lalu
satu PERTANYAAN di bagian paling bawah. Jawablah dari BAHAN itu.

Dua kegagalan yang sama beratnya, dan kamu harus menghindari dua-duanya:

  MENGARANG  -- menjawab dari ingatanmu sendiri, bukan dari BAHAN.
  MENGABAIKAN -- jawabanmu jauh lebih miskin daripada BAHAN yang kamu pegang.

Yang kedua lebih sering terjadi dan lebih sulit ketahuan. Kalau BAHAN memuat
lima butir dan kamu menulis "beberapa pihak tertentu", kamu sudah gagal --
sekalipun tidak ada satu kata pun yang salah.


# CARA MEMBACA BAHAN

Tiap kutipan diawali nomor dalam kurung siku, lalu identitas peraturannya,
alamat pasalnya, baru bunyinya:

    [3] PERATURAN MENTERI KEUANGAN Nomor 81 Tahun 2024
        tentang Ketentuan Perpajakan ...
        Pasal 1 > angka 12

    <bunyi pasalnya>

Penanda yang bisa muncul, dan artinya:

    [PERUBAHAN] di baris alamat
        Teks ini BUKAN norma milik peraturan di kepala blok. Ini bunyi baru
        yang dipasang peraturan itu ke peraturan LAIN. Jangan menyebut
        peraturan di kepala blok sebagai sumber normanya.

    baris diawali ###
        Bukan bunyi peraturan, melainkan keterangan sistem tentang riwayat
        sebuah pasal: dokumen asalnya apa, sudah diubah berapa kali, oleh
        siapa, dan mana yang berlaku. WAJIB dibaca dan dipakai, tapi jangan
        dikutip seolah-olah bunyi pasal.

    [BERLAKU]        versi yang berlaku sekarang -- ini yang dipakai menjawab
    [BERLAKU PADA t] versi yang berlaku pada tahun yang ditanyakan. ITU yang
                     diminta -- jangan diganti dengan yang terbaru, dan jangan
                     bilang penanya salah
    [SEBELUM]        bunyi sebelum perubahan yang ditanyakan
    [SESUDAH]        bunyi sesudah perubahan yang ditanyakan
    [ASLI]           bunyi mula-mula, SUDAH TIDAK BERLAKU
    [PERUBAHAN n]    bunyi antara, SUDAH TIDAK BERLAKU
    ### !!           peringatan bahwa data sistem tidak lengkap

    Kalau baris ### menyebut versi lain yang TIDAK ikut dikirim (nama dan
    tanggalnya saja), versi itu tetap boleh -- dan kadang wajib -- kamu sebut
    sebagai keterangan. Yang tidak boleh: mengarang bunyinya.


# LANGKAH KERJA

1. Baca SELURUH blok sebelum menulis apa pun. Jangan berhenti di blok pertama
   yang kelihatan cocok.
2. Tentukan apa yang sebenarnya diminta: definisi? angka? syarat? daftar?
   batas waktu? siapa yang wajib? pengecualiannya?
3. Kumpulkan SEMUA blok yang menyumbang jawaban -- syarat, pengecualian, dan
   sanksi sering berada di pasal yang berbeda dari pokoknya.
4. Kalau sebuah pasal punya beberapa versi, pakai yang [BERLAKU] -- KECUALI
   BAHAN memang mengirim lebih dari satu versi. [SEBELUM]/[SESUDAH] dan
   [ASLI]/[PERUBAHAN n] dikirim justru supaya kamu PAKAI SEMUANYA; sistem
   tidak pernah mengirim versi lama tanpa alasan.
5. Tulis jawabannya lengkap, lalu dasar hukumnya, lalu catatan kalau ada.


# ATURAN

## 1. Hanya dari BAHAN

Kamu mungkin merasa tahu jawabannya. Itu ingatan dari data latihanmu, dan di
sini tidak berlaku: aturan pajak berubah, dan BAHAN inilah yang sudah
diperiksa tanggalnya.

Jawaban benar yang tidak bersumber dari BAHAN tetap kegagalan: sistem ini jadi
tidak bisa dinilai, dan lain kali dia akan mengarang di tempat yang tidak kamu
sadari.

## 2. Jawab TUNTAS -- ini aturan terpenting di sini

Kedalaman jawabanmu mengikuti kekayaan BAHAN, bukan seleramu.

- BAHAN menyebut daftar? Tulis SELURUH butirnya, jangan diringkas jadi
  kategori.
- BAHAN memuat pengecualian, syarat, atau batas? Sebutkan.
- Beberapa blok saling melengkapi? Gabungkan jadi satu jawaban utuh, jangan
  pilih satu lalu buang sisanya.
- Satu blok merujuk "sebagaimana dimaksud dalam Pasal X", dan Pasal X ada di
  BAHAN? Ikuti rujukannya, pakai isinya.

Contoh kegagalan yang nyata terjadi, jangan diulangi:

    Pertanyaan  : "Siapa saja yang wajib memungut PPh Pasal 22?"
    BAHAN       : sembilan blok, salah satunya memuat daftar pemungut
                  huruf a sampai h, blok lain memuat pengecualiannya

    SALAH -> "Badan-badan tertentu yang ditetapkan oleh Menteri Keuangan."
    BENAR -> daftar pemungutnya satu per satu persis seperti di BAHAN,
             lalu pengecualiannya, lalu dasar hukum tiap bagian

Versi SALAH itu tidak memuat satu pun kata yang keliru. Dia tetap gagal,
karena penanya tidak jadi tahu apa pun yang belum dia tahu sebelum bertanya.

Tapi tuntas itu relatif ke yang DIMINTA. Diminta ringkas, tuntas berarti
ringkasan yang tidak menyesatkan plus satu kalimat tentang apa yang belum ikut
disebut -- bukan alasan untuk tetap menyalin semuanya.

## 3. Jangan menambah yang tidak tertulis

Angka, batas waktu, persentase, dan syarat DISALIN. Tidak dibulatkan, tidak
dikira-kira dari "yang biasanya".

    BENAR -> paling lama 3 (tiga) bulan sejak ...     [tertulis di BAHAN]
    SALAH -> sekitar 3 bulan                          [dihaluskan]
    SALAH -> 3 bulan, dan bisa diperpanjang           [tidak ada di BAHAN]

Tuntas bukan berarti melebar. Lengkapi dari BAHAN, jangan dari ingatan.

Membandingkan bukan menambah. Kalau BAHAN memuat dua versi dari pasal yang
sama, menyebut bagian mana yang berubah itu WAJIB -- selama dua-duanya kamu
tunjuk di BAHAN. Yang dilarang menyimpulkan dari ingatan, bukan menyimpulkan
dari BAHAN.

## 4. Sitasi: nomor blok, dan dasar hukum yang DISALIN

Tiap klaim harus bisa dilacak. Sebut nomor blok yang kamu pakai, misalnya
"[4]", di dekat klaimnya.

Untuk dasar hukum: kalau ada baris "### bunyi yang berlaku: ...", salin
kalimat itu apa adanya. Kalimat itu sudah disusun sistem memakai bentuk
penyebutan resmi.

    BENAR -> Pasal 1 PER-57/PJ/2010 sebagaimana telah beberapa kali diubah,
             terakhir dengan PER-31/PJ/2015  [4]
    SALAH -> PER-31/PJ/2015 Pasal 1

Versi SALAH menyembunyikan peraturan induknya. Orang yang mencari
"PER-31/PJ/2015 Pasal 1" tidak akan menemukan konteksnya, dan tidak akan tahu
pasal itu sebenarnya milik siapa.

Kalau baris "### bunyi yang berlaku" tidak ada, sebut nama peraturan dan nomor
pasal persis seperti tertulis di kepala blok.

## 5. Peringatan "### !!" wajib disampaikan

Terjemahkan ke bahasa biasa, taruh di akhir jawaban. Jangan didiamkan supaya
jawabanmu kelihatan rapi -- penanya berhak tahu bagian mana yang belum pasti.

Yang paling sering: riwayat perubahan tidak lengkap karena peraturan
terkaitnya belum ada di korpus. Artinya kamu TIDAK BOLEH bilang "tidak pernah
diubah". Yang benar: "tidak ditemukan perubahannya di bahan yang tersedia".
Dua kalimat itu terdengar mirip dan artinya jauh berbeda.

Kalau tidak ada "### !!" sama sekali, hilangkan seluruh baris Catatan. Jangan
menulis bahwa tidak ada peringatan -- itu bukan informasi.

## 6. Pertanyaan kabur: jawab, jangan balik bertanya

Kamu tidak punya giliran kedua. Kalau pertanyaannya bisa dibaca beberapa cara,
jawab bacaan yang paling didukung BAHAN, lalu sebutkan asumsimu dalam satu
kalimat. Kalau BAHAN mendukung dua-duanya, jawab dua-duanya terpisah.

## 7. Ikuti bahasa penanya

Ditanya santai, jawab santai. Ditanya formal, jawab formal. Tapi nomor pasal
dan nama peraturan SELALU lengkap dan resmi, apa pun gayanya. Istilah teknis
yang tidak bisa dihindari dijelaskan sekali dengan bahasa sehari-hari.


# KALAU BAHANNYA TIDAK CUKUP

Jangan memaksakan. Tulis:

    Tidak ada di bahan yang tersedia.

lalu sebutkan blok mana yang paling mendekati dan apa isinya, supaya penanya
tahu harus mencari ke mana. Ini jawaban yang sah, bukan kegagalan.

Kalau BAHAN menjawab SEBAGIAN, jawab bagian itu selengkap mungkin, lalu
sebutkan dengan jelas bagian mana yang belum terjawab. Jangan menutupi
lubangnya dengan kalimat umum yang terdengar lengkap.


# BENTUK JAWABAN

Langsung ke jawabannya. Tanpa pengantar semacam "Berdasarkan bahan yang
diberikan" atau mengulang pertanyaannya.

Satu kalimat yang mengatur sisanya:

    PERTANYAAN menentukan BENTUK. BAHAN menentukan ISI. Jangan ditukar.

Yang TETAP, bentuk apa pun:
  - isinya dari BAHAN
  - angka, batas waktu, dan persentase DISALIN
  - dasar hukum di akhir
  - peringatan "### !!" disampaikan

Selain empat itu, ikuti pertanyaannya. Yang di bawah CONTOH, bukan daftar
tertutup:

  "bandingkan" / "bedanya apa" / "sebelum-sesudah"
      Sebut dulu apa yang berubah, 1-3 kalimat, tunjuk ayat atau hurufnya.
      Baru kutip bagian yang berubah dari tiap versi. Bagian yang TIDAK
      berubah cukup disebut, tidak perlu dikutip ulang.

  "di ayat berapa" / "pasal mana"
      Alamatnya duluan, baru potongan bunyinya. Jangan menyalin satu pasal
      penuh untuk menjawab pertanyaan satu baris.

  "singkatnya" / "ringkas" / "intinya apa"
      Ringkas beneran. Angka tetap disalin. Tutup dengan satu kalimat:
      bagian mana yang masih ada rinciannya kalau mau digali.

  "berapa" / "kapan" / "sampai kapan"
      Angka atau tanggalnya di kalimat PERTAMA. Penjelasannya menyusul.

  "boleh tidak" / "wajib tidak" / "kena pajak tidak"
      Ya atau tidak dulu, baru dasarnya. Kalau BAHAN tidak tegas, bilang
      tidak tegas -- jangan dipaksa jadi ya/tidak.

  "jelaskan ke orang awam"
      Bahasa sehari-hari. Istilah resminya ditaruh dalam kurung sekali.

  "buatkan tabel" / "buat daftar"
      Turuti. Sitasi tetap masuk -- di kolom sendiri atau di bawah tabel.

Kalau bentuk yang diminta tidak ada di daftar ini: TIRU bentuk pertanyaannya.
Ditanya satu kalimat, jawab padat. Ditanya bertingkat, jawab bertingkat.
Jangan balik ke rangka panjang cuma karena itu yang paling aman.

Rangka default, dipakai kalau pertanyaannya tidak meminta bentuk khusus:

    <jawaban -- selengkap yang BAHAN dukung. Pakai daftar bernomor atau
     berbutir kalau isinya memang daftar. Sebut nomor blok di dekat klaim.>

    Dasar hukum:
    - <disalin dari BAHAN>  [nomor blok]

    Catatan: <hanya kalau ada baris ### !!, atau ada asumsi yang kamu ambil.
              Kalau tidak ada, hilangkan seluruh baris ini.>

Kutip bunyi pasal seperlunya -- bagian yang menentukan jawabannya, bukan
seluruh pasal, dan bukan pula cuma ringkasannya.
"""

### 9.2 bahan(), tampil_rencana(), jawab() -- pipeline utuh

In [26]:
def bahan(hasil, rantai):
    """Konteks pencarian + riwayat versi, penomoran blok menyambung.

    Dipisah dari jawab() supaya eval bisa memeriksa bahannya tanpa membakar
    satu panggilan LLM.
    """
    teks = konteks(hasil)
    if rantai:
        teks += PEMISAH + blok_versi(rantai, mulai=len(hasil) + 1)
    return teks


def tampil_rencana(paham):
    """Keputusan planner, dicetak SEBELUM retrieval jalan.

    Urutannya disengaja: jejak pencarian di bawahnya (dokumen apa yang
    diresolusi, saringan tingkat berapa yang kepakai) cuma bisa dibaca kalau
    kamu sudah tahu kunci apa yang dikirim ke sana.
    """
    s = paham["strategi"]
    print(f"-- rencana  ({paham['_model']}, {paham['_detik']}s)")
    print(f"   tipe        : {paham['tipe']}")
    print(f"   peraturan   : {paham['peraturan'] or '-'}")
    print(f"   pasal       : {', '.join(paham['pasal']) or '-'}")
    print(f"   waktu       : {s['temporal']}"
          f"{'  (tahun ' + paham['tahun'] + ')' if paham['tahun'] else ''}")
    print(f"   rewritten   : {paham['rewritten_query']}")
    print(f"   sub_queries :")
    for i, q in enumerate(paham["sub_queries"], 1):
        print(f"      {i}. {q}")
    print(f"   cakupan     : {paham['cakupan']}  -> {s['k']} blok")
    print(f"   pemeringkat : {s['pemeringkat']}")
    print(f"   saringan    : peraturan={s['saringan']['peraturan'] or '-'}"
          f"  pasal={s['saringan']['pasal'] or '-'}")
    print()


def jawab(pertanyaan, k=None, cetak=True, perencana=None, penjawab=None):
    """Pipeline utuh: pertanyaan -> jawaban. -> dict berisi semua tahapnya.

    Dua penyedia dipisah dengan sengaja. Tugasnya berbeda jauh: planner harus
    patuh pada skema JSON dan boleh singkat; penjawab harus menulis panjang,
    rapi, dan setia pada kutipan.
    """
    perencana = MODEL_PLANNER if perencana is None else perencana
    penjawab = MODEL_PENJAWAB if penjawab is None else penjawab

    paham = planner(pertanyaan, penyedia=perencana)
    if cetak:
        tampil_rencana(paham)
    hasil = retrieve(paham, pertanyaan, k=k)
    rantai = ekspansi(hasil, paham)
    bersih = rapikan(hasil, rantai)
    # BAHAN dulu, PERTANYAAN paling bawah: bahan bisa 9 blok penuh teks pasal,
    # dan pertanyaan yang terkubur di atas tumpukan itu lebih mudah terlupa
    # daripada yang menempel di titik model mulai menulis.
    isi = bahan(bersih, rantai)
    isi = f"# BAHAN\n\n{isi}\n\n{PEMISAH}\n# PERTANYAAN\n\n{pertanyaan}"
    teks, detik, nama = panggil_llm(SISTEM_JAWAB, None, isi, penyedia=penjawab)

    r = {"pertanyaan": pertanyaan, "jawaban": teks, "paham": paham,
         "hasil": bersih, "rantai": rantai, "bahan": isi,
         "_model_planner": paham["_model"], "_model_jawab": nama,
         "_detik_planner": paham["_detik"], "_detik_jawab": round(detik, 2)}
    if cetak:
        garis = "=" * 70
        print(f"\n{garis}\nQ: {pertanyaan}")
        print(f"   planner {r['_detik_planner']}s  ({r['_model_planner']})")
        print(f"   jawab   {r['_detik_jawab']}s  ({r['_model_jawab']})")
        print(f"   bahan   {len(bersih)} blok + {len(rantai)} rantai versi")
        for x in rantai:
            print(f"           riwayat {x['pasal']}: {len(x['semua'])} versi"
                  f"  ({x['sebab']}, {x['mode']})")
        print(f"{garis}\n")
        print(teks)
    return r

#### cek: penomoran blok bahan menyambung

In [27]:
def _uji_bahan():
    """Penomoran blok tidak boleh loncat atau dobel -- kalau `mulai` salah,
    LLM melihat dua blok bernomor sama dan rujukannya jadi ambigu."""
    p57 = _id_dok("PER-57/PJ/2010", "2010")
    versi, sebab = riwayat(p57, "Pasal 1")
    rantai = [{"pasal": "Pasal 1", "sebab": sebab, "akar": p57,
               "versi": versi, "semua": versi, "mode": "all"}]
    hasil = rapikan(_di(p57, "Pasal 2"))
    assert hasil, "PER-57/PJ/2010 Pasal 2 harus ada di indeks"

    nomor = [int(n) for n in re.findall(r"^\[(\d+)\]", bahan(hasil, rantai), re.M)]
    assert nomor == list(range(1, len(nomor) + 1)), nomor
    assert len(nomor) == len(hasil) + len(versi), (len(nomor), len(hasil), len(versi))
    print(f"ok  bahan (penomoran {nomor[0]}..{nomor[-1]} menyambung)")


_uji_bahan()

ok  bahan (penomoran 1..4 menyambung)


### Coba-coba

#### Muat ulang .env (kalau kunci/model baru diubah)

In [28]:
ENV = muat_env()

#### Jalankan satu pertanyaan

In [29]:
# -- model sel ini. Bawaannya ikut knop 1.2, timpa di sini kalau mau beda.
PAKAI_PLANNER = ""     # "" = lokal
PAKAI_JAWAB = "Groq"     

# PERTANYAAN = "kalau saya nggak terima hasil keberatan, saya harus ke mana dan batas waktunya berapa lama?"
PERTANYAAN = "Apa saja persyaratan untuk menjadi P3I menurut KEP-238/PJ/2012"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

planner  : google/gemma-4-12b-qat   penjawab : openai/gpt-oss-120b
-- rencana  (google/gemma-4-12b-qat, 32.69s)
   tipe        : prosedur
   peraturan   : ['KEP-238/PJ/2012']
   pasal       : -
   waktu       : latest
   rewritten   : persyaratan untuk menjadi Perusahaan Penunjang Jasa Keuangan (P3I) berdasarkan Keputusan Direktur Jenderal Pajak Nomor KEP-238/PJ/2012
   sub_queries :
      1. persyaratan menjadi Perusahaan Penunjang Jasa Keuangan (P3I)
      2. syarat administrasi P3I KEP-238/PJ/2012
      3. ketentuan P3I dalam Keputusan Direktur Jenderal Pajak Nomor KEP-238/PJ/2012
   cakupan     : sedang  -> 6 blok
   pemeringkat : dense
   saringan    : peraturan=['KEP-238/PJ/2012']  pasal=-

  dokumen: 'KEP-238/PJ/2012' -> keputusan-direktur-jenderal-pajak-kep-238-pj-2012-db628   [tabel]
  4 kunci x 50 -> 16 calon unik  [saringan: dokumen saja, ambil 6]

Q: Apa saja persyaratan untuk menjadi P3I menurut KEP-238/PJ/2012
   planner 32.69s  (google/gemma-4-12b-qat)
   jawab   1.64s  

In [30]:
print(r["bahan"])

# BAHAN

[1] Keputusan Direktur Jenderal Pajak Nomor KEP-238/PJ/2012 Tahun 2012
    tentang PENERAPAN PENGENDALIAN INTERN DI LINGKUNGAN DIREKTORAT JENDERAL PAJAK
    KELIMA

KELIMA UP3I sampai dengan terbentuknya UKI yang permanen, terdiri dari:
  a. Kepala UP3I: 1) untuk tingkat kantor pusat adalah Direktur Kepatuhan Internal dan Transformasi Daya Aparatur (KITSDA); 2) untuk tingkat kantor wilayah adalah Kepala Bagian Umum; 3) untuk tingkat Kantor Pelayanan Pajak (KPP) adalah Kepala Subbagian Umum;
  b. Anggota UP3I: 1) untuk tingkat kantor pusat adalah Pelaksana Pemantauan Pengendalian Intern (P3I) pada Subdirektorat Kepatuhan Internal Direktorat KITSDA; 2) untuk tingkat kantor wilayah adalah P3I pada kantor wilayah; 3) untuk tingkat KPP adalah P3I pada KPP.

----------------------------------------------------------------------

[2] Keputusan Direktur Jenderal Pajak Nomor KEP-238/PJ/2012 Tahun 2012
    tentang PENERAPAN PENGENDALIAN INTERN DI LINGKUNGAN DIREKTORAT JENDERAL PAJAK
   

#### Rencana planner + rantai versi

In [31]:
# Rencana planner + rantai versi, kalau mau lihat keputusannya
print(json.dumps({k: v for k, v in r["paham"].items() if not k.startswith("_")},
                 indent=2, ensure_ascii=False))

for x in r["rantai"]:
    print()
    print(x['pasal'], '|', len(x['semua']), 'versi | sebab:', x['sebab'], '| mode:', x['mode'])


{
  "peraturan": [
    "KEP-238/PJ/2012"
  ],
  "pasal": [],
  "temporal_mode": "latest",
  "tahun": "",
  "tipe": "prosedur",
  "cakupan": "sedang",
  "rewritten_query": "persyaratan untuk menjadi Perusahaan Penunjang Jasa Keuangan (P3I) berdasarkan Keputusan Direktur Jenderal Pajak Nomor KEP-238/PJ/2012",
  "sub_queries": [
    "persyaratan menjadi Perusahaan Penunjang Jasa Keuangan (P3I)",
    "syarat administrasi P3I KEP-238/PJ/2012",
    "ketentuan P3I dalam Keputusan Direktur Jenderal Pajak Nomor KEP-238/PJ/2012"
  ],
  "strategi": {
    "pemeringkat": "dense",
    "saringan": {
      "peraturan": [
        "KEP-238/PJ/2012"
      ],
      "pasal": []
    },
    "temporal": "latest",
    "k": 6
  }
}


## 10. Tes

#### Muat ulang .env

In [32]:
ENV = muat_env()

#### Jalankan satu pertanyaan (penjawab = Groq)

In [34]:
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "OpenAI"   

PERTANYAAN = "Apa saja perubahan ketentuan Pajak Penghasilan dalam UU Nomor 36 Tahun 2008 setelah diubah melalui UU Nomor 7 Tahun 2021, khususnya terkait tarif PPh Orang Pribadi?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

planner  : google/gemma-4-12b-qat   penjawab : gpt-5.6-luna
-- rencana  (google/gemma-4-12b-qat, 105.32s)
   tipe        : perubahan
   peraturan   : ['UU Nomor 36 Tahun 2008', 'UU Nomor 7 Tahun 2021']
   pasal       : -
   waktu       : diff
   rewritten   : perubahan ketentuan Pajak Penghasilan pada Undang-Undang Nomor 36 Tahun 2008 setelah diubah dengan Undang-Undang Nomor 7 Tahun 2021 khususnya mengenai tarif Pajak Penghasilan Orang Pribadi
   sub_queries :
      1. perubahan tarif Pajak Penghasilan Orang Pribadi dalam Undang-Undang Nomor 7 Tahun 2021
      2. ketentuan Pajak Penghasilan Orang Pribadi dalam Undang-Undang Nomor 36 Tahun 2008
      3. perbandingan tarif Pajak Penghasilan Orang Pribadi sebelum dan sesudah Undang-Undang Nomor 7 Tahun 2021
   cakupan     : sedang  -> 6 blok
   pemeringkat : lexical
   saringan    : peraturan=['UU Nomor 36 Tahun 2008', 'UU Nomor 7 Tahun 2021']  pasal=-

  dokumen: 'UU Nomor 36 Tahun 2008' -> undang-undang-36-tahun-2008-db9916   [tabel]
 

In [ ]:
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Pasal 2 KMK 231/KMK.03/2001 sudah diubah berapa kali dan oleh peraturan apa saja?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Bunyi Pasal 2 KMK 231/KMK.03/2001 yang berlaku pada tahun 2016 seperti apa?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#4
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Pasal 2 KMK 231/KMK.03/2001 tahun 1999 bunyinya apa?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#5
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa yang berubah pada Pasal 2 KMK 231/KMK.03/2001 setelah diubah PMK 198/PMK.010/2019?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#6
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Tunjukkan semua versi Pasal 3 KMK 516/KMK.04/2000 dari awal sampai sekarang"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#7
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa isi Pasal 19A KEP-07/BC/2003?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#8
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apakah KEP-220/PJ/2002 masih berlaku sekarang?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

### B. Alamat & saringan

In [ ]:
#9
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa isi Diktum keempat belas KEP-238/PJ/2012?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#10
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa isi diktum kesatu KEP-238/PJ/2012?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#10b
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa isi diktum pertama KEP-238/PJ/2012?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#11
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa definisi Pengusaha Kena Pajak menurut Pasal 1 UU PPN?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#12
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa isi Pasal 56 dan Pasal 57 PP 55 tahun 2022?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#13
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa isi Pasal 3B PER-57/PJ/2010?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

### C. Resolusi dokumen

In [ ]:
#14
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa sanksi kalau terlambat lapor SPT Tahunan menurut UU KUP?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#15
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa isi Pasal 4 PMK 81/2024?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#16
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Peraturan mana yang mengatur PPh atas penghasilan dari pengalihan hak atas tanah dan bangunan?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#17
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa isi Pasal 12 PMK 999/PMK.03/2099?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

### D. Tipe & cakupan

In [ ]:
#18
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Berapa tarif PPh final UMKM dan sampai kapan bisa dipakai?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#19
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Kalau saya nggak terima hasil keberatan, saya harus ke mana dan batas waktunya berapa lama?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#20
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Tenaga kerja asing dengan keahlian tertentu perlakuan PPh-nya bagaimana? Diatur di PP dan PMK yang mana?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#21
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Apa bedanya definisi 'Pengusaha Kena Pajak' di UU PPN dengan di PP 44/2022?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])

In [ ]:
#22
PAKAI_PLANNER = ""     
PAKAI_JAWAB = "Groq"   

PERTANYAAN = "Siapa saja yang ditunjuk sebagai pemungut PPh Pasal 22?"

print(f"planner  : {sumber(PAKAI_PLANNER)[1]}   penjawab : {sumber(PAKAI_JAWAB)[1]}")

r = jawab(PERTANYAAN, perencana=PAKAI_PLANNER, penjawab=PAKAI_JAWAB)

print(r["bahan"])